In [ ]:
# Imports

import io
import os
import re
import json
import base64
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from pymongo import MongoClient

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

In [ ]:
# ============================================================
# CONFIG
# ============================================================

DATASET_DIR = Path(
    r"C:\Users\re29faj\Desktop\ebl_tablets_and_sign_crops\sign_crops"
)

BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)


In [ ]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ============================================================
# TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),

    transforms.RandomRotation(7),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.92, 1.08),
        shear=3
    ),

    transforms.ColorJitter(
        brightness=0.12,
        contrast=0.15
    ),

    transforms.RandomGrayscale(p=0.15),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

test_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [ ]:
# ============================================================
# BUILD DATASET WITH PROGRESS
# label = signName_period
# ============================================================

samples = []

period_dirs = [p for p in DATASET_DIR.iterdir() if p.is_dir()]

for period_dir in tqdm(period_dirs, desc="Scanning periods"):

    period = period_dir.name
    sign_dirs = [s for s in period_dir.iterdir() if s.is_dir()]

    for sign_dir in tqdm(sign_dirs, desc=f"Scanning signs in {period}", leave=False):

        sign_name = sign_dir.name
        label = f"{sign_name}_{period}"

        image_paths = list(sign_dir.glob("*.png"))

        for img_path in image_paths:
            samples.append({
                "image_path": str(img_path),
                "label": label,
                "period": period,
                "sign_name": sign_name,
            })

df = pd.DataFrame(samples)

print("Total samples:", len(df))

In [ ]:
# ============================================================
# REMOVE RARE CLASSES
# Ensure safe 65/15/20 split
# ============================================================

MIN_TRAIN = 65
MIN_VAL = 15
MIN_TEST = 20

MIN_SAMPLES_PER_CLASS = (
    MIN_TRAIN + MIN_VAL + MIN_TEST
)

print("Minimum samples per class required:",
      MIN_SAMPLES_PER_CLASS)

class_counts = df["label"].value_counts()

valid_classes = class_counts[
    class_counts >= MIN_SAMPLES_PER_CLASS
].index.tolist()

rare_classes = class_counts[
    class_counts < MIN_SAMPLES_PER_CLASS
]

print("\nRare classes removed:",
      len(rare_classes))

print("Remaining valid classes:",
      len(valid_classes))

df = df[
    df["label"].isin(valid_classes)
].reset_index(drop=True)

print("\nRemaining samples:", len(df))
print("Remaining classes:", df["label"].nunique())


In [ ]:
# ============================================================
# LABEL ENCODING
# ============================================================

labels_sorted = sorted(df["label"].unique())

label_to_idx = {
    label: idx
    for idx, label in enumerate(labels_sorted)
}

idx_to_label = {
    idx: label
    for label, idx in label_to_idx.items()
}

df["label_idx"] = df["label"].map(label_to_idx)


In [ ]:
# ============================================================
# TRAIN / VAL / TEST SPLIT
# 65 / 15 / 20
# ============================================================

# First split:
# 65% train
# 35% remaining

train_df, temp_df = train_test_split(
    df,
    test_size=0.35,
    stratify=df["label_idx"],
    random_state=RANDOM_SEED
)

# Second split:
# Remaining 35% -> 15% val + 20% test
# Therefore:
# validation fraction inside temp = 15 / 35
# test fraction inside temp = 20 / 35

val_df, test_df = train_test_split(
    temp_df,
    test_size=(20 / 35),
    stratify=temp_df["label_idx"],
    random_state=RANDOM_SEED
)

print("\nDataset split:")
print("Train:", len(train_df), f"({len(train_df)/len(df):.2%})")
print("Val  :", len(val_df), f"({len(val_df)/len(df):.2%})")
print("Test :", len(test_df), f"({len(test_df)/len(df):.2%})")
print("\nMinimum samples per class after split:")
print("Train:", train_df["label"].value_counts().min())
print("Val  :", val_df["label"].value_counts().min())
print("Test :", test_df["label"].value_counts().min())

In [ ]:
# ============================================================
# DATASET CLASS
# ============================================================

class SignDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_path = row["image_path"]   # change if your column has another name
        label = row["label_idx"]       # change if your label column has another name

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label, img_path

In [ ]:
# ============================================================
# DATALOADERS
# ============================================================

train_dataset = SignDataset(train_df, transform=train_transform)
val_dataset = SignDataset(val_df, transform=test_transform)
test_dataset = SignDataset(test_df, transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)

In [ ]:
# ============================================================
# CNN ConvNeXt MODELS TO EVALUATE
# ============================================================
from torchvision import models

MODEL_DICT = {
    "convnext_tiny": (
        models.convnext_tiny,
        models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
    ),

    "convnext_base": (
        models.convnext_base,
        models.ConvNeXt_Base_Weights.IMAGENET1K_V1
    ),
}

In [ ]:
# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

# ============================================================
# TOP-K ACCURACY
# ============================================================

def top_k_accuracy(outputs, targets, k=1):

    with torch.no_grad():

        _, pred = outputs.topk(k, dim=1)

        correct = pred.eq(
            targets.view(-1, 1).expand_as(pred)
        )

        correct_total = correct.sum().item()

        return correct_total / targets.size(0)

In [ ]:
# ============================================================
# TRAIN ALL CONVNEXT MODELS WITH HISTORY + EARLY STOPPING + CSV + GRAPHS
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = False

# ============================================================
# CNN MODELS TO EVALUATE
# ============================================================

MODEL_DICT = {
    "convnext_tiny": (
        models.convnext_tiny,
        models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
    ),

    "convnext_base": (
        models.convnext_base,
        models.ConvNeXt_Base_Weights.IMAGENET1K_V1
    ),
}

# ============================================================
# TOP-K ACCURACY
# ============================================================

def top_k_accuracy(outputs, targets, k=1):

    with torch.no_grad():

        _, pred = outputs.topk(k, dim=1)

        correct = pred.eq(
            targets.view(-1, 1).expand_as(pred)
        )

        correct_total = correct.sum().item()

        return correct_total / targets.size(0)


# ============================================================
# STORE FINAL MODEL COMPARISON
# ============================================================

all_model_results = []

# ============================================================
# TRAIN MODELS ONE BY ONE
# ============================================================

for MODEL_NAME, (MODEL_CLASS, MODEL_WEIGHTS) in MODEL_DICT.items():

    print("\n" + "=" * 70)
    print(f"Training model: {MODEL_NAME}")
    print("=" * 70)

    model = MODEL_CLASS(
        weights=MODEL_WEIGHTS
    )

    # ========================================================
    # REPLACE CONVNEXT CLASSIFICATION HEAD
    # ========================================================

    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model = model.to(DEVICE)

    # =========================
    # LOSS + OPTIMIZER
    # =========================

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    # =========================
    # RESET HISTORY FOR THIS MODEL
    # =========================

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_top1": [],
        "train_top2": [],
        "train_top3": [],
        "val_top1": [],
        "val_top2": [],
        "val_top3": [],
    }

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = 0

    patience = 5
    patience_counter = 0

    # ========================================================
    # TRAINING LOOP
    # ========================================================

    for epoch in range(NUM_EPOCHS):

        # =========================
        # TRAIN
        # =========================

        model.train()

        train_loss = 0.0
        train_top1 = 0.0
        train_top2 = 0.0
        train_top3 = 0.0
        train_batches = 0

        train_pbar = tqdm(
            train_loader,
            desc=f"{MODEL_NAME} | Epoch {epoch+1}/{NUM_EPOCHS}"
        )

        for images, labels, paths in train_pbar:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_top1 += top_k_accuracy(outputs, labels, k=1)
            train_top2 += top_k_accuracy(outputs, labels, k=2)
            train_top3 += top_k_accuracy(outputs, labels, k=3)

            train_batches += 1

            train_pbar.set_postfix({
                "loss": f"{loss.item():.4f}"
            })

        train_loss /= train_batches
        train_top1 /= train_batches
        train_top2 /= train_batches
        train_top3 /= train_batches

        # =========================
        # VALIDATION
        # =========================

        model.eval()

        val_loss = 0.0
        val_top1 = 0.0
        val_top2 = 0.0
        val_top3 = 0.0
        val_batches = 0

        with torch.no_grad():

            for images, labels, paths in tqdm(
                val_loader,
                desc=f"{MODEL_NAME} | Validation {epoch+1}/{NUM_EPOCHS}"
            ):

                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                val_top1 += top_k_accuracy(outputs, labels, k=1)
                val_top2 += top_k_accuracy(outputs, labels, k=2)
                val_top3 += top_k_accuracy(outputs, labels, k=3)

                val_batches += 1

        val_loss /= val_batches
        val_top1 /= val_batches
        val_top2 /= val_batches
        val_top3 /= val_batches

        # =========================
        # SAVE HISTORY
        # =========================

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        history["train_top1"].append(train_top1)
        history["train_top2"].append(train_top2)
        history["train_top3"].append(train_top3)

        history["val_top1"].append(val_top1)
        history["val_top2"].append(val_top2)
        history["val_top3"].append(val_top3)

        # =========================
        # PRINT RESULTS
        # =========================

        print("\n")
        print(f"Model      : {MODEL_NAME}")
        print(f"Epoch      : {epoch+1}/{NUM_EPOCHS}")
        print(f"Train Loss : {train_loss:.4f}")
        print(f"Val Loss   : {val_loss:.4f}")
        print(f"Train Top-1: {train_top1:.4f}")
        print(f"Train Top-2: {train_top2:.4f}")
        print(f"Train Top-3: {train_top3:.4f}")
        print(f"Val Top-1  : {val_top1:.4f}")
        print(f"Val Top-2  : {val_top2:.4f}")
        print(f"Val Top-3  : {val_top3:.4f}")

        # =========================
        # SAVE BEST MODEL + EARLY STOPPING
        # =========================

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_val_acc = val_top1
            best_epoch = epoch + 1

            torch.save(
                model.state_dict(),
                f"best_{MODEL_NAME}_sign_classifier.pth"
            )

            patience_counter = 0
            print(f"Saved best {MODEL_NAME} model based on validation loss.")

        else:

            patience_counter += 1

            print(
                f"No validation-loss improvement. "
                f"Patience: {patience_counter}/{patience}"
            )

            if patience_counter >= patience:
                print(f"\nEarly stopping triggered for {MODEL_NAME}.")
                break

    # ========================================================
    # SAVE HISTORY CSV FOR THIS MODEL
    # ========================================================

    num_completed_epochs = len(history["train_loss"])

    history_df = pd.DataFrame(history)
    history_df.insert(0, "epoch", range(1, num_completed_epochs + 1))

    history_csv_path = f"{MODEL_NAME}_training_history.csv"
    history_df.to_csv(history_csv_path, index=False)

    print(f"Saved training history: {history_csv_path}")

    # ========================================================
    # SAVE TRAINING CURVES FOR THIS MODEL
    # ========================================================

    epochs = range(1, num_completed_epochs + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.axvline(
        history["val_loss"].index(min(history["val_loss"])) + 1,
        linestyle="--",
        label="Best Val Loss"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{MODEL_NAME}: Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{MODEL_NAME}_loss_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_top1"], label="Train Top-1")
    plt.plot(epochs, history["val_top1"], label="Validation Top-1")
    plt.xlabel("Epoch")
    plt.ylabel("Top-1 Accuracy")
    plt.title(f"{MODEL_NAME}: Top-1 Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{MODEL_NAME}_top1_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_top2"], label="Train Top-2")
    plt.plot(epochs, history["val_top2"], label="Validation Top-2")
    plt.xlabel("Epoch")
    plt.ylabel("Top-2 Accuracy")
    plt.title(f"{MODEL_NAME}: Top-2 Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{MODEL_NAME}_top2_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_top3"], label="Train Top-3")
    plt.plot(epochs, history["val_top3"], label="Validation Top-3")
    plt.xlabel("Epoch")
    plt.ylabel("Top-3 Accuracy")
    plt.title(f"{MODEL_NAME}: Top-3 Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{MODEL_NAME}_top3_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    # ========================================================
    # STORE SUMMARY RESULT
    # ========================================================

    all_model_results.append({
        "model": MODEL_NAME,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_top1": best_val_acc,
        "best_val_top2": max(history["val_top2"]),
        "best_val_top3": max(history["val_top3"]),
        "completed_epochs": num_completed_epochs
    })

    # Free GPU memory before next model
    del model
    torch.cuda.empty_cache()


# ============================================================
# SAVE FINAL MODEL COMPARISON CSV
# ============================================================

model_comparison_df = pd.DataFrame(all_model_results)
model_comparison_df.to_csv("convnext_model_comparison.csv", index=False)

print("\nSaved final model comparison: convnext_model_comparison.csv")
print(model_comparison_df)

In [ ]:
# ============================================================
# TEST EVALUATION FOR CONVNEXT BASE MODEL ONLY
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm.auto import tqdm

# ============================================================
# MODEL DICTIONARY
# ============================================================

MODEL_DICT = {
    "convnext_base": models.convnext_base,
}

# ============================================================
# CHECKPOINT PATHS
# ============================================================

CKPT_DICT = {
    "convnext_base": "best_convnext_base_sign_classifier.pth",
}

# ============================================================
# TEST ALL SELECTED MODELS
# ============================================================

all_test_results = []

for MODEL_NAME, MODEL_CLASS in MODEL_DICT.items():

    print("\n" + "=" * 60)
    print(f"Evaluating {MODEL_NAME} on test set...")
    print("=" * 60)

    # =========================
    # CREATE MODEL
    # =========================

    model = MODEL_CLASS(weights=None)

    # ConvNeXt classification head
    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model = model.to(DEVICE)

    # =========================
    # LOAD BEST CHECKPOINT
    # =========================

    CKPT_PATH = CKPT_DICT[MODEL_NAME]

    model.load_state_dict(
        torch.load(
            CKPT_PATH,
            map_location=DEVICE
        )
    )

    model.eval()

    all_labels = []
    all_preds = []

    top1_acc = 0.0
    top2_acc = 0.0
    top3_acc = 0.0
    num_batches = 0

    # =========================
    # TEST LOOP
    # =========================

    with torch.no_grad():

        for batch in tqdm(test_loader, desc=f"Testing {MODEL_NAME}"):

            if len(batch) == 2:
                images, labels = batch
            else:
                images, labels, paths = batch

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            top1_acc += top_k_accuracy(outputs, labels, k=1)
            top2_acc += top_k_accuracy(outputs, labels, k=2)
            top3_acc += top_k_accuracy(outputs, labels, k=3)

            num_batches += 1

    top1_acc /= num_batches
    top2_acc /= num_batches
    top3_acc /= num_batches

    precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    print("\n================ TEST RESULTS ================\n")
    print(f"Model          : {MODEL_NAME}")
    print(f"Top-1 Accuracy : {top1_acc:.4f}")
    print(f"Top-2 Accuracy : {top2_acc:.4f}")
    print(f"Top-3 Accuracy : {top3_acc:.4f}")
    print(f"Precision      : {precision:.4f}")
    print(f"Recall         : {recall:.4f}")
    print(f"Macro-F1       : {macro_f1:.4f}")
    print("\n==============================================")

    # =========================
    # SAVE INDIVIDUAL CSV
    # =========================

    result = {
        "model": MODEL_NAME,
        "top1": top1_acc,
        "top2": top2_acc,
        "top3": top3_acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "macro_f1": macro_f1,
    }

    results_df = pd.DataFrame([result])

    csv_path = f"{MODEL_NAME}_test_results.csv"
    results_df.to_csv(csv_path, index=False)

    print("Saved test results:", csv_path)

    all_test_results.append(result)

    # =========================
    # PLOT TEST METRICS
    # =========================

    test_metrics = {
        "Top-1": top1_acc,
        "Top-2": top2_acc,
        "Top-3": top3_acc,
        "Precision": precision,
        "Recall": recall,
        "Macro-F1": macro_f1,
    }

    metric_names = list(test_metrics.keys())
    metric_values = list(test_metrics.values())

    plt.figure(figsize=(10, 5))
    bars = plt.bar(metric_names, metric_values)

    plt.ylim(0, 1.0)
    plt.ylabel("Score")
    plt.title(f"Test Metrics - {MODEL_NAME}")

    for bar, value in zip(bars, metric_values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.01,
            f"{value:.3f}",
            ha="center"
        )

    plt.grid(axis="y")
    plt.savefig(
        f"{MODEL_NAME}_test_metrics.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    del model
    torch.cuda.empty_cache()


# ============================================================
# SAVE FINAL TEST COMPARISON CSV
# ============================================================

test_comparison_df = pd.DataFrame(all_test_results)
test_comparison_df.to_csv("convnext_base_test_comparison.csv", index=False)

print("\nSaved final test comparison: convnext_base_test_comparison.csv")
print(test_comparison_df)

In [ ]:
# ============================================================
# GRAD-CAM VISUALIZATION FOR CONVNEXT BASE MODEL
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ============================================================
# MODEL DICTIONARY
# ============================================================

MODEL_DICT = {
    "convnext_base": models.convnext_base,
}

CKPT_DICT = {
    "convnext_base": "best_convnext_base_sign_classifier.pth",
}

# ============================================================
# IMAGE TRANSFORM
# ============================================================

cam_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# ============================================================
# SELECT IMAGE FROM TEST SET
# ============================================================

sample_idx = 90

img_path = test_df.iloc[sample_idx]["image_path"]
true_label_idx = test_df.iloc[sample_idx]["label_idx"]

image = Image.open(img_path).convert("RGB")

input_tensor = cam_transform(image).unsqueeze(0).to(DEVICE)

# RGB image for visualization
rgb_img = image.resize((224, 224))
rgb_img = np.array(rgb_img).astype(np.float32) / 255.0

# ============================================================
# RUN GRAD-CAM FOR CONVNEXT BASE
# ============================================================

for MODEL_NAME, MODEL_CLASS in MODEL_DICT.items():

    print("\n" + "=" * 60)
    print(f"Grad-CAM for {MODEL_NAME}")
    print("=" * 60)

    # ----------------------------
    # Load model
    # ----------------------------

    model = MODEL_CLASS(weights=None)

    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model.load_state_dict(
        torch.load(
            CKPT_DICT[MODEL_NAME],
            map_location=DEVICE
        )
    )

    model = model.to(DEVICE)
    model.eval()

    # ----------------------------
    # Prediction
    # ----------------------------

    with torch.no_grad():

        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)

        pred_idx = outputs.argmax(dim=1).item()
        pred_conf = probs[0, pred_idx].item()

    print("Image:", img_path)
    print("True label index:", true_label_idx)
    print("Pred label index:", pred_idx)
    print("Confidence:", pred_conf)

    # ----------------------------
    # Grad-CAM target layer
    # ----------------------------

    # ConvNeXt Base final convolutional block
    target_layers = [
        model.features[-1][-1]
    ]

    cam = GradCAM(
        model=model,
        target_layers=target_layers
    )

    targets = [
        ClassifierOutputTarget(pred_idx)
    ]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

    cam_image = show_cam_on_image(
        rgb_img,
        grayscale_cam,
        use_rgb=True
    )

    # ----------------------------
    # Plot
    # ----------------------------

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(rgb_img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(cam_image)
    plt.title(f"{MODEL_NAME} Grad-CAM")
    plt.axis("off")

    plt.tight_layout()

    plt.savefig(
        f"{MODEL_NAME}_gradcam_sample_{sample_idx}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    del model
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# FIND CORRECT AND WRONG TEST CASES FOR CONVNEXT BASE MODEL
# ============================================================

import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from tqdm.auto import tqdm

# ============================================================
# MODEL DICTIONARY
# ============================================================

MODEL_DICT = {
    "convnext_base": models.convnext_base,
}

# ============================================================
# CHECKPOINT PATHS
# ============================================================

CKPT_DICT = {
    "convnext_base": "best_convnext_base_sign_classifier.pth",
}

# ============================================================
# CREATE idx_to_label IF NOT ALREADY CREATED
# ============================================================

idx_to_label = {
    v: k for k, v in label_to_idx.items()
}

# ============================================================
# FIND CASES MODEL BY MODEL
# ============================================================

for MODEL_NAME, MODEL_CLASS in MODEL_DICT.items():

    print("\n" + "=" * 60)
    print(f"Finding correct/wrong cases for {MODEL_NAME}")
    print("=" * 60)

    # =========================
    # LOAD MODEL
    # =========================

    model = MODEL_CLASS(weights=None)

    # ConvNeXt classifier head
    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model.load_state_dict(
        torch.load(
            CKPT_DICT[MODEL_NAME],
            map_location=DEVICE
        )
    )

    model = model.to(DEVICE)
    model.eval()

    # =========================
    # COLLECT PREDICTIONS
    # =========================

    cases = []

    with torch.no_grad():

        for batch in tqdm(
            test_loader,
            desc=f"Testing {MODEL_NAME}"
        ):

            if len(batch) == 2:
                images, labels = batch
                paths = [""] * len(labels)
            else:
                images, labels, paths = batch

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            pred_probs, preds = torch.max(
                probs,
                dim=1
            )

            for path, true_idx, pred_idx, conf in zip(
                paths,
                labels.cpu().numpy(),
                preds.cpu().numpy(),
                pred_probs.cpu().numpy()
            ):

                true_label = idx_to_label[int(true_idx)]
                pred_label = idx_to_label[int(pred_idx)]

                cases.append({
                    "model": MODEL_NAME,
                    "image_path": path,
                    "true_idx": int(true_idx),
                    "pred_idx": int(pred_idx),
                    "true_label": true_label,
                    "pred_label": pred_label,
                    "confidence": float(conf),
                    "correct": int(true_idx) == int(pred_idx)
                })

    # =========================
    # SAVE CASES
    # =========================

    cases_df = pd.DataFrame(cases)

    csv_path = f"{MODEL_NAME}_test_correct_wrong_cases.csv"
    cases_df.to_csv(csv_path, index=False)

    correct_count = cases_df["correct"].sum()
    wrong_count = len(cases_df) - correct_count

    print(f"Saved: {csv_path}")
    print(f"Total cases : {len(cases_df)}")
    print(f"Correct     : {correct_count}")
    print(f"Wrong       : {wrong_count}")

    # =========================
    # SAVE USEFUL SUBSETS
    # =========================

    correct_cases = cases_df[cases_df["correct"] == True]
    wrong_cases = cases_df[cases_df["correct"] == False]

    correct_cases.sort_values(
        "confidence",
        ascending=False
    ).head(50).to_csv(
        f"{MODEL_NAME}_high_confidence_correct_cases.csv",
        index=False
    )

    wrong_cases.sort_values(
        "confidence",
        ascending=False
    ).head(50).to_csv(
        f"{MODEL_NAME}_high_confidence_wrong_cases.csv",
        index=False
    )

    wrong_cases.sort_values(
        "confidence",
        ascending=True
    ).head(50).to_csv(
        f"{MODEL_NAME}_low_confidence_wrong_cases.csv",
        index=False
    )

    print(f"Saved high-confidence correct cases for {MODEL_NAME}")
    print(f"Saved high-confidence wrong cases for {MODEL_NAME}")
    print(f"Saved low-confidence wrong cases for {MODEL_NAME}")

    del model
    torch.cuda.empty_cache()

In [ ]:
cases_df = pd.read_csv("convnext_base_test_correct_wrong_cases.csv")
#cases_df = pd.read_csv("resnet50_test_correct_wrong_cases.csv")
#cases_df = pd.read_csv("resnet101_test_correct_wrong_cases.csv")


correct_cases = cases_df[cases_df["correct"] == True]
wrong_cases = cases_df[cases_df["correct"] == False]

display(correct_cases.sort_values("confidence", ascending=False).head(10))
display(wrong_cases.sort_values("confidence", ascending=False).head(10))

In [ ]:
# ============================================================
# GRAD-CAM FOR CORRECT AND WRONG CASES — CONVNEXT BASE
# ============================================================

# Install once if needed:
# pip install grad-cam

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ============================================================
# SETTINGS
# ============================================================

MODEL_NAME = "convnext_base"

CKPT_PATH = f"best_{MODEL_NAME}_sign_classifier.pth"
CASES_CSV = f"{MODEL_NAME}_test_correct_wrong_cases.csv"

OUTPUT_DIR = f"{MODEL_NAME}_gradcam_correct_wrong"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CORRECT = 5
NUM_WRONG = 5

# ============================================================
# LOAD MODEL
# ============================================================

MODEL_DICT = {
    "convnext_base": models.convnext_base,
}

model = MODEL_DICT[MODEL_NAME](weights=None)

num_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(
    num_features,
    len(label_to_idx)
)

model.load_state_dict(
    torch.load(
        CKPT_PATH,
        map_location=DEVICE
    )
)

model = model.to(DEVICE)
model.eval()

# ============================================================
# TRANSFORM
# ============================================================

cam_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# ============================================================
# LOAD CASES
# ============================================================

cases_df = pd.read_csv(CASES_CSV)

correct_cases = (
    cases_df[cases_df["correct"] == True]
    .sort_values("confidence", ascending=False)
    .head(NUM_CORRECT)
)

wrong_cases = (
    cases_df[cases_df["correct"] == False]
    .sort_values("confidence", ascending=False)
    .head(NUM_WRONG)
)

selected_cases = pd.concat([
    correct_cases.assign(case_type="correct"),
    wrong_cases.assign(case_type="wrong")
])

# ============================================================
# GRAD-CAM FUNCTION
# ============================================================

def make_gradcam(row, idx):

    img_path = row["image_path"]
    true_label = row["true_label"]
    pred_label = row["pred_label"]
    pred_idx = int(row["pred_idx"])
    confidence = float(row["confidence"])
    case_type = row["case_type"]

    image = Image.open(img_path).convert("RGB")

    input_tensor = cam_transform(image).unsqueeze(0).to(DEVICE)

    rgb_img = image.resize((224, 224))
    rgb_img = np.array(rgb_img).astype(np.float32) / 255.0

    # ConvNeXt Base final convolutional block
    target_layers = [
        model.features[-1][-1]
    ]

    cam = GradCAM(
        model=model,
        target_layers=target_layers
    )

    targets = [
        ClassifierOutputTarget(pred_idx)
    ]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

    cam_image = show_cam_on_image(
        rgb_img,
        grayscale_cam,
        use_rgb=True
    )

    plt.figure(figsize=(9, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(rgb_img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(cam_image)
    plt.title(
        f"{case_type.upper()}\n"
        f"True: {true_label}\n"
        f"Pred: {pred_label}\n"
        f"Conf: {confidence:.3f}"
    )
    plt.axis("off")

    plt.tight_layout()

    save_path = os.path.join(
        OUTPUT_DIR,
        f"{MODEL_NAME}_{case_type}_{idx}_gradcam.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Saved:", save_path)


# ============================================================
# RUN GRAD-CAM
# ============================================================

for idx, row in selected_cases.reset_index(drop=True).iterrows():
    make_gradcam(row, idx)

In [ ]:
# ============================================================
# CELL 1 — LOAD METADATA CSV
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

METADATA_CSV = r"C:\Users\re29faj\Desktop\ebl_tablets_and_sign_crops\metadata\crop_metadata.csv"

df = pd.read_csv(
    METADATA_CSV,
    encoding="utf-8-sig"
)

print("CSV columns:")
print(df.columns.tolist())

print("\nNumber of rows:", len(df))

df.head()

In [ ]:
# ============================================================
# CELL 2 — NORMALIZE CROP PATHS
# ============================================================

def norm_path(p):
    return str(p).replace("\\", "/").lower().strip()

df["cropPath_norm"] = df["cropPath"].apply(norm_path)

df[[
    "cropPath",
    "cropPath_norm",
    "fragmentNumber",
    "period",
    "signName"
]].head()

In [ ]:
# ============================================================
# EXTRACT CONVNEXT BASE EMBEDDINGS
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from tqdm.auto import tqdm

# ============================================================
# MODEL DICTIONARY
# ============================================================

MODEL_DICT = {
    "convnext_base": models.convnext_base,
}

CKPT_DICT = {
    "convnext_base": "best_convnext_base_sign_classifier.pth",
}

# ============================================================
# CONVNEXT EMBEDDING FUNCTION
# ============================================================

def extract_convnext_embedding(model, images):

    x = model.features(images)
    x = model.avgpool(x)
    x = torch.flatten(x, 1)

    return x


# ============================================================
# EXTRACT EMBEDDINGS MODEL BY MODEL
# ============================================================

for MODEL_NAME, MODEL_CLASS in MODEL_DICT.items():

    print("\n" + "=" * 60)
    print(f"Extracting embeddings for {MODEL_NAME}")
    print("=" * 60)

    # =========================
    # CREATE MODEL
    # =========================

    model = MODEL_CLASS(weights=None)

    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model = model.to(DEVICE)

    # =========================
    # LOAD CHECKPOINT
    # =========================

    ckpt_path = CKPT_DICT[MODEL_NAME]

    model.load_state_dict(
        torch.load(
            ckpt_path,
            map_location=DEVICE
        )
    )

    model.eval()

    all_embeddings = []
    all_labels = []
    all_paths = []

    # =========================
    # EXTRACT
    # =========================

    with torch.no_grad():

        for batch in tqdm(
            test_loader,
            desc=f"Extracting {MODEL_NAME} embeddings"
        ):

            if len(batch) == 2:
                images, labels = batch
                paths = [""] * len(labels)
            else:
                images, labels, paths = batch

            images = images.to(DEVICE)

            embeddings = extract_convnext_embedding(
                model,
                images
            )

            all_embeddings.append(
                embeddings.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_paths.extend(paths)

    all_embeddings = np.vstack(all_embeddings)
    all_labels = np.array(all_labels)

    print("Embeddings shape:", all_embeddings.shape)
    print("Labels shape:", all_labels.shape)
    print("Number of paths:", len(all_paths))

    if len(all_paths) > 0:
        print("Example path:", all_paths[0])

    # =========================
    # SAVE EMBEDDINGS
    # =========================

    np.save(
        f"{MODEL_NAME}_test_embeddings.npy",
        all_embeddings
    )

    np.save(
        f"{MODEL_NAME}_test_labels.npy",
        all_labels
    )

    paths_df = pd.DataFrame({
        "image_path": all_paths,
        "label_idx": all_labels
    })

    paths_df.to_csv(
        f"{MODEL_NAME}_test_embedding_paths.csv",
        index=False
    )

    print(f"Saved: {MODEL_NAME}_test_embeddings.npy")
    print(f"Saved: {MODEL_NAME}_test_labels.npy")
    print(f"Saved: {MODEL_NAME}_test_embedding_paths.csv")

    del model
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# CELL 4 — MATCH EMBEDDINGS WITH CSV METADATA
# ============================================================

# ============================================================
# MATCH SAVED EMBEDDINGS WITH CSV METADATA FOR CONVNEXT BASE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# NORMALIZE PATH FUNCTION
# ============================================================

def norm_path(p):
    return str(Path(p)).replace("\\", "/").lower()

# ============================================================
# PREPARE METADATA DF
# ============================================================

df["cropPath_norm"] = df["cropPath"].apply(norm_path)

metadata_cols = [
    "cropPath_norm",
    "fragmentNumber",
    "period",
    "signName",
    "x",
    "y",
    "width",
    "height",
    "imagePath",
    "txtPath"
]

# ============================================================
# PROCESS CONVNEXT BASE
# ============================================================

MODEL_NAMES = [
    "convnext_base"
]

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"Matching metadata for {MODEL_NAME}")
    print("=" * 60)

    # Load saved labels and paths
    labels = np.load(f"{MODEL_NAME}_test_labels.npy")

    paths_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_paths.csv"
    )

    emb_df = pd.DataFrame({
        "cropPath": paths_df["image_path"],
        "label_id": labels
    })

    emb_df["cropPath_norm"] = emb_df["cropPath"].apply(norm_path)

    emb_df = emb_df.merge(
        df[metadata_cols],
        on="cropPath_norm",
        how="left"
    )

    print("Missing values after merge:")
    print(
        emb_df[["fragmentNumber", "period", "signName"]]
        .isna()
        .sum()
    )

    output_csv = f"{MODEL_NAME}_test_embedding_metadata.csv"
    emb_df.to_csv(output_csv, index=False)

    print("Saved:", output_csv)
    display(emb_df.head())

In [ ]:
# ============================================================
# SIGN PURITY AND TABLET PURITY FOR CONVNEXT BASE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

k = 10

MODEL_NAMES = [
    "convnext_base"
]

purity_results = []

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"Computing purity for {MODEL_NAME}")
    print("=" * 60)

    # =========================
    # LOAD EMBEDDINGS + METADATA
    # =========================

    embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    emb_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    # =========================
    # VALID ROWS ONLY
    # =========================

    valid_mask = (
        emb_df["signName"].notna() &
        emb_df["fragmentNumber"].notna()
    )

    embeddings_valid = embeddings[valid_mask.values]

    sign_names = emb_df.loc[
        valid_mask,
        "signName"
    ].values

    fragment_numbers = emb_df.loc[
        valid_mask,
        "fragmentNumber"
    ].values

    print("Valid embeddings:", embeddings_valid.shape)

    # =========================
    # NEAREST NEIGHBORS
    # =========================

    nn = NearestNeighbors(
        n_neighbors=k + 1,
        metric="cosine"
    )

    nn.fit(embeddings_valid)

    distances, indices = nn.kneighbors(
        embeddings_valid
    )

    # remove self-neighbor
    neighbor_indices = indices[:, 1:]

    # =========================
    # PURITY SCORES
    # =========================

    sign_purity_scores = []
    tablet_purity_scores = []

    for i in range(len(embeddings_valid)):

        neighbor_signs = sign_names[
            neighbor_indices[i]
        ]

        neighbor_fragments = fragment_numbers[
            neighbor_indices[i]
        ]

        sign_purity_scores.append(
            np.mean(neighbor_signs == sign_names[i])
        )

        tablet_purity_scores.append(
            np.mean(neighbor_fragments == fragment_numbers[i])
        )

    sign_purity = np.mean(sign_purity_scores)
    tablet_purity = np.mean(tablet_purity_scores)

    print(f"Sign Purity   : {sign_purity:.4f}")
    print(f"Tablet Purity : {tablet_purity:.4f}")

    purity_results.append({
        "model": MODEL_NAME,
        "k": k,
        "sign_purity": sign_purity,
        "tablet_purity": tablet_purity,
        "valid_samples": len(embeddings_valid)
    })

# ============================================================
# SAVE RESULTS
# ============================================================

purity_df = pd.DataFrame(purity_results)

purity_df.to_csv(
    "convnext_base_sign_tablet_purity.csv",
    index=False
)

print("\nSaved: convnext_base_sign_tablet_purity.csv")
print(purity_df)

In [ ]:
# ============================================================
# SAVE METADATA + PURITY FOR CONVNEXT BASE
# ============================================================

MODEL_NAMES = [
    "convnext_base"
]

for MODEL_NAME in MODEL_NAMES:

    # Load metadata already created earlier
    emb_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    # Load purity row for this model
    summary = purity_df[
        purity_df["model"] == MODEL_NAME
    ]

    summary.to_csv(
        f"{MODEL_NAME}_purity_results.csv",
        index=False
    )

    print(f"Saved: {MODEL_NAME}_purity_results.csv")

In [ ]:
# ============================================================
# SAME-SIGN SIMILARITY ANALYSIS FOR CONVNEXT BASE
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import random

MODEL_NAMES = [
    "convnext_base"
]

results = []

max_pairs_per_sign = 500

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"Analyzing {MODEL_NAME}")
    print("=" * 60)

    # ========================================================
    # LOAD EMBEDDINGS + METADATA
    # ========================================================

    embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    emb_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    # ========================================================
    # VALID ENTRIES
    # ========================================================

    valid_mask = (
        emb_df["signName"].notna() &
        emb_df["fragmentNumber"].notna()
    )

    df_valid = emb_df.loc[
        valid_mask
    ].reset_index(drop=True)

    emb_valid = embeddings[
        valid_mask.values
    ]

    # ========================================================
    # SIMILARITIES
    # ========================================================

    same_tablet_sims = []
    different_tablet_sims = []

    for sign in tqdm(
        df_valid["signName"].unique(),
        desc=f"{MODEL_NAME}"
    ):

        idx = df_valid.index[
            df_valid["signName"] == sign
        ].tolist()

        if len(idx) < 2:
            continue

        n_pairs = min(
            max_pairs_per_sign,
            len(idx) * (len(idx) - 1) // 2
        )

        for _ in range(n_pairs):

            i, j = random.sample(idx, 2)

            sim = cosine_similarity(
                emb_valid[i].reshape(1, -1),
                emb_valid[j].reshape(1, -1)
            )[0, 0]

            if (
                df_valid.loc[i, "fragmentNumber"]
                ==
                df_valid.loc[j, "fragmentNumber"]
            ):
                same_tablet_sims.append(sim)
            else:
                different_tablet_sims.append(sim)

    # ========================================================
    # RESULTS
    # ========================================================

    same_mean = np.mean(same_tablet_sims)
    diff_mean = np.mean(different_tablet_sims)

    print(
        f"Same sign + same tablet      : {same_mean:.4f}"
    )

    print(
        f"Same sign + different tablet : {diff_mean:.4f}"
    )

    print(
        f"Difference                   : "
        f"{same_mean - diff_mean:.4f}"
    )

    results.append({
        "model": MODEL_NAME,
        "same_sign_same_tablet": same_mean,
        "same_sign_diff_tablet": diff_mean,
        "difference": same_mean - diff_mean
    })

# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    "convnext_base_same_sign_similarity.csv",
    index=False
)

print("\nSaved: convnext_base_same_sign_similarity.csv")
print(results_df)

In [ ]:
# ============================================================
# SAME-SIGN SIMILARITY ANALYSIS + DISTRIBUTION PLOTS
# CONVNEXT BASE
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

MODEL_NAMES = [
    "convnext_base"
]

max_pairs_per_sign = 500

results = []
similarity_distributions = {}

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"Analyzing {MODEL_NAME}")
    print("=" * 60)

    embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    emb_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    valid_mask = (
        emb_df["signName"].notna() &
        emb_df["fragmentNumber"].notna()
    )

    df_valid = emb_df.loc[
        valid_mask
    ].reset_index(drop=True)

    emb_valid = embeddings[
        valid_mask.values
    ]

    same_tablet_sims = []
    different_tablet_sims = []

    for sign in tqdm(
        df_valid["signName"].unique(),
        desc=f"{MODEL_NAME}"
    ):

        idx = df_valid.index[
            df_valid["signName"] == sign
        ].tolist()

        if len(idx) < 2:
            continue

        n_pairs = min(
            max_pairs_per_sign,
            len(idx) * (len(idx) - 1) // 2
        )

        for _ in range(n_pairs):

            i, j = random.sample(idx, 2)

            sim = cosine_similarity(
                emb_valid[i].reshape(1, -1),
                emb_valid[j].reshape(1, -1)
            )[0, 0]

            if (
                df_valid.loc[i, "fragmentNumber"]
                ==
                df_valid.loc[j, "fragmentNumber"]
            ):
                same_tablet_sims.append(sim)
            else:
                different_tablet_sims.append(sim)

    same_mean = np.mean(same_tablet_sims)
    diff_mean = np.mean(different_tablet_sims)

    results.append({
        "model": MODEL_NAME,
        "same_sign_same_tablet": same_mean,
        "same_sign_diff_tablet": diff_mean,
        "difference": same_mean - diff_mean
    })

    similarity_distributions[MODEL_NAME] = {
        "same": same_tablet_sims,
        "different": different_tablet_sims
    }

    print(f"Same sign + same tablet      : {same_mean:.4f}")
    print(f"Same sign + different tablet : {diff_mean:.4f}")
    print(f"Difference                   : {same_mean - diff_mean:.4f}")


# ============================================================
# SAVE NUMERIC RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    "convnext_base_same_sign_similarity.csv",
    index=False
)

print("\nSaved: convnext_base_same_sign_similarity.csv")
print(results_df)


# ============================================================
# PLOT HISTOGRAM DISTRIBUTIONS
# ============================================================

for MODEL_NAME in similarity_distributions.keys():

    same_sims = similarity_distributions[MODEL_NAME]["same"]
    diff_sims = similarity_distributions[MODEL_NAME]["different"]

    plt.figure(figsize=(8, 5))

    plt.hist(
        same_sims,
        bins=50,
        density=True,
        alpha=0.6,
        label="Same sign + same tablet"
    )

    plt.hist(
        diff_sims,
        bins=50,
        density=True,
        alpha=0.6,
        label="Same sign + different tablet"
    )

    plt.xlabel("Cosine Similarity")
    plt.ylabel("Density")
    plt.title(f"{MODEL_NAME}: Same-Sign Similarity Distribution")
    plt.legend()
    plt.grid(True)

    plt.savefig(
        f"{MODEL_NAME}_similarity_distribution.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# PLOT MEAN SIMILARITY COMPARISON
# ============================================================

x = np.arange(len(results_df))
width = 0.35

plt.figure(figsize=(8, 5))

plt.bar(
    x - width / 2,
    results_df["same_sign_same_tablet"],
    width,
    label="Same sign + same tablet"
)

plt.bar(
    x + width / 2,
    results_df["same_sign_diff_tablet"],
    width,
    label="Same sign + different tablet"
)

plt.xticks(
    x,
    results_df["model"]
)

plt.ylabel("Mean Cosine Similarity")
plt.title("Same-Sign Similarity Across Tablets")
plt.legend()
plt.grid(axis="y")

plt.savefig(
    "convnext_base_same_sign_similarity_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# CREATE FRAGMENT / TABLET EMBEDDINGS FOR CONVNEXT BASE
# ============================================================

import numpy as np
import pandas as pd

MODEL_NAMES = [
    "convnext_base"
]

MIN_SIGNS_PER_FRAGMENT = 5

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"Creating fragment embeddings for {MODEL_NAME}")
    print("=" * 60)

    # ========================================================
    # LOAD EMBEDDINGS + METADATA
    # ========================================================

    embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    emb_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    # ========================================================
    # VALID ENTRIES ONLY
    # ========================================================

    valid_mask = (
        emb_df["fragmentNumber"].notna() &
        emb_df["signName"].notna()
    )

    df_valid = emb_df.loc[
        valid_mask
    ].reset_index(drop=True)

    emb_valid = embeddings[
        valid_mask.values
    ]

    # ========================================================
    # CREATE FRAGMENT EMBEDDINGS
    # ========================================================

    fragment_embeddings = {}
    fragment_info = []

    for fragment in df_valid["fragmentNumber"].unique():

        idx = df_valid.index[
            df_valid["fragmentNumber"] == fragment
        ].tolist()

        if len(idx) < MIN_SIGNS_PER_FRAGMENT:
            continue

        fragment_embedding = emb_valid[idx].mean(axis=0)

        fragment_embeddings[fragment] = fragment_embedding

        fragment_info.append({
            "fragmentNumber": fragment,
            "num_signs": len(idx),
            "period": df_valid.loc[idx, "period"].mode()[0]
        })

    fragment_info_df = pd.DataFrame(
        fragment_info
    )

    # ========================================================
    # SAVE
    # ========================================================

    fragment_names = np.array(
        list(fragment_embeddings.keys())
    )

    fragment_vectors = np.vstack(
        list(fragment_embeddings.values())
    )

    np.save(
        f"{MODEL_NAME}_fragment_embeddings.npy",
        fragment_vectors
    )

    np.save(
        f"{MODEL_NAME}_fragment_names.npy",
        fragment_names
    )

    fragment_info_df.to_csv(
        f"{MODEL_NAME}_fragment_info.csv",
        index=False
    )

    print(
        "Fragments:",
        len(fragment_embeddings)
    )

    print(
        "Embedding shape:",
        fragment_vectors.shape
    )

    print(
        "Saved:",
        f"{MODEL_NAME}_fragment_embeddings.npy"
    )

    print(
        "Saved:",
        f"{MODEL_NAME}_fragment_names.npy"
    )

    print(
        "Saved:",
        f"{MODEL_NAME}_fragment_info.csv"
    )

    display(fragment_info_df.head())

In [ ]:
# ============================================================
# EFFICIENT FRAGMENT NEAREST-NEIGHBOR SIMILARITY
# CONVNEXT BASE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

MODEL_NAMES = [
    "convnext_base"
]

TOP_K = 10

all_neighbor_results = []

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"Finding nearest fragments for {MODEL_NAME}")
    print("=" * 60)

    fragment_matrix = np.load(
        f"{MODEL_NAME}_fragment_embeddings.npy"
    )

    fragment_ids = np.load(
        f"{MODEL_NAME}_fragment_names.npy",
        allow_pickle=True
    )

    fragment_info_df = pd.read_csv(
        f"{MODEL_NAME}_fragment_info.csv"
    )

    nn = NearestNeighbors(
        n_neighbors=TOP_K + 1,
        metric="cosine",
        algorithm="brute"
    )

    nn.fit(fragment_matrix)

    distances, indices = nn.kneighbors(fragment_matrix)

    rows = []

    for i, fragment_id in enumerate(fragment_ids):

        for rank in range(1, TOP_K + 1):

            neighbor_idx = indices[i, rank]
            distance = distances[i, rank]
            similarity = 1.0 - distance

            rows.append({
                "model": MODEL_NAME,
                "query_fragment": fragment_id,
                "neighbor_fragment": fragment_ids[neighbor_idx],
                "rank": rank,
                "cosine_similarity": similarity
            })

    neighbors_df = pd.DataFrame(rows)

    neighbors_df = neighbors_df.merge(
        fragment_info_df.rename(columns={
            "fragmentNumber": "query_fragment",
            "period": "query_period",
            "num_signs": "query_num_signs"
        }),
        on="query_fragment",
        how="left"
    )

    neighbors_df = neighbors_df.merge(
        fragment_info_df.rename(columns={
            "fragmentNumber": "neighbor_fragment",
            "period": "neighbor_period",
            "num_signs": "neighbor_num_signs"
        }),
        on="neighbor_fragment",
        how="left"
    )

    neighbors_df["same_period"] = (
        neighbors_df["query_period"] ==
        neighbors_df["neighbor_period"]
    )

    output_csv = f"{MODEL_NAME}_fragment_top{TOP_K}_neighbors.csv"

    neighbors_df.to_csv(
        output_csv,
        index=False
    )

    print("Saved:", output_csv)
    print(neighbors_df.head())

    all_neighbor_results.append(neighbors_df)

all_neighbors_df = pd.concat(
    all_neighbor_results,
    ignore_index=True
)

all_neighbors_df.to_csv(
    f"convnext_base_fragment_top{TOP_K}_neighbors_all_models.csv",
    index=False
)

print("\nSaved combined file:")
print(f"convnext_base_fragment_top{TOP_K}_neighbors_all_models.csv")

In [ ]:
# ============================================================
# TOP SIMILAR FRAGMENTS + SHARED-SIGN SIMILARITY
# CONVNEXT BASE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

MODEL_NAMES = [
    "convnext_base"
]

QUERY_FRAGMENT = "K.1283"
TOP_K = 20
MIN_SHARED_SIGNS = 3


# ============================================================
# SHARED-SIGN FRAGMENT SIMILARITY FUNCTION
# ============================================================

def shared_sign_fragment_similarity(
    fragment_a,
    fragment_b,
    df_valid,
    emb_valid,
    min_shared_signs=3
):
    df_a = df_valid[df_valid["fragmentNumber"] == fragment_a]
    df_b = df_valid[df_valid["fragmentNumber"] == fragment_b]

    signs_a = set(df_a["signName"])
    signs_b = set(df_b["signName"])

    shared_signs = signs_a.intersection(signs_b)

    if len(shared_signs) < min_shared_signs:
        return None

    sign_scores = []

    for sign in shared_signs:

        idx_a = df_a.index[df_a["signName"] == sign].tolist()
        idx_b = df_b.index[df_b["signName"] == sign].tolist()

        emb_a = emb_valid[idx_a]
        emb_b = emb_valid[idx_b]

        sim_matrix = cosine_similarity(
            emb_a,
            emb_b
        )

        sign_scores.append(
            sim_matrix.mean()
        )

    return {
        "fragmentA": fragment_a,
        "fragmentB": fragment_b,
        "num_shared_signs": len(shared_signs),
        "shared_sign_similarity": float(np.mean(sign_scores))
    }


# ============================================================
# RUN FOR CONVNEXT BASE
# ============================================================

all_query_results = []

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"Processing {MODEL_NAME}")
    print("=" * 60)

    # ========================================================
    # LOAD FRAGMENT EMBEDDINGS
    # ========================================================

    fragment_matrix = np.load(
        f"{MODEL_NAME}_fragment_embeddings.npy"
    )

    fragment_ids = np.load(
        f"{MODEL_NAME}_fragment_names.npy",
        allow_pickle=True
    )

    fragment_info_df = pd.read_csv(
        f"{MODEL_NAME}_fragment_info.csv"
    )

    if QUERY_FRAGMENT not in fragment_ids:
        print(f"{QUERY_FRAGMENT} not found for {MODEL_NAME}")
        continue

    # ========================================================
    # FIND TOP-K SIMILAR FRAGMENTS
    # ========================================================

    nn = NearestNeighbors(
        n_neighbors=TOP_K + 1,
        metric="cosine",
        algorithm="brute"
    )

    nn.fit(fragment_matrix)

    query_idx = np.where(fragment_ids == QUERY_FRAGMENT)[0][0]

    distances, indices = nn.kneighbors(
        fragment_matrix[query_idx].reshape(1, -1)
    )

    rows = []

    for rank in range(1, TOP_K + 1):

        neighbor_idx = indices[0, rank]
        similarity = 1.0 - distances[0, rank]

        rows.append({
            "model": MODEL_NAME,
            "queryFragment": QUERY_FRAGMENT,
            "candidateFragment": fragment_ids[neighbor_idx],
            "rank": rank,
            "fragment_embedding_similarity": similarity
        })

    top_candidates_df = pd.DataFrame(rows)

    top_candidates_df = top_candidates_df.merge(
        fragment_info_df.rename(columns={
            "fragmentNumber": "candidateFragment",
            "period": "candidate_period",
            "num_signs": "candidate_num_signs"
        }),
        on="candidateFragment",
        how="left"
    )

    query_info = fragment_info_df[
        fragment_info_df["fragmentNumber"] == QUERY_FRAGMENT
    ]

    if len(query_info) > 0:
        top_candidates_df["query_period"] = query_info.iloc[0]["period"]
        top_candidates_df["query_num_signs"] = query_info.iloc[0]["num_signs"]

    # ========================================================
    # LOAD SIGN-LEVEL EMBEDDINGS FOR SHARED-SIGN SIMILARITY
    # ========================================================

    sign_embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    emb_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    valid_mask = (
        emb_df["fragmentNumber"].notna() &
        emb_df["signName"].notna()
    )

    df_valid = emb_df.loc[
        valid_mask
    ].reset_index(drop=True)

    emb_valid = sign_embeddings[
        valid_mask.values
    ]

    # ========================================================
    # COMPUTE SHARED-SIGN SIMILARITY FOR TOP CANDIDATES
    # ========================================================

    shared_results = []

    for candidate in top_candidates_df["candidateFragment"]:

        result = shared_sign_fragment_similarity(
            QUERY_FRAGMENT,
            candidate,
            df_valid,
            emb_valid,
            min_shared_signs=MIN_SHARED_SIGNS
        )

        if result is None:

            shared_results.append({
                "candidateFragment": candidate,
                "num_shared_signs": 0,
                "shared_sign_similarity": np.nan
            })

        else:

            shared_results.append({
                "candidateFragment": candidate,
                "num_shared_signs": result["num_shared_signs"],
                "shared_sign_similarity": result["shared_sign_similarity"]
            })

    shared_df = pd.DataFrame(shared_results)

    top_candidates_df = top_candidates_df.merge(
        shared_df,
        on="candidateFragment",
        how="left"
    )

    top_candidates_df["same_period"] = (
        top_candidates_df["query_period"] ==
        top_candidates_df["candidate_period"]
    )

    # ========================================================
    # SAVE MODEL-SPECIFIC RESULT
    # ========================================================

    output_csv = (
        f"{MODEL_NAME}_{QUERY_FRAGMENT}_top{TOP_K}_fragment_candidates.csv"
    )

    top_candidates_df.to_csv(
        output_csv,
        index=False
    )

    print("Saved:", output_csv)
    display(top_candidates_df)

    all_query_results.append(top_candidates_df)


# ============================================================
# SAVE COMBINED RESULT
# ============================================================

if len(all_query_results) > 0:

    all_query_results_df = pd.concat(
        all_query_results,
        ignore_index=True
    )

    combined_csv = (
        f"convnext_base_{QUERY_FRAGMENT}_top{TOP_K}_fragment_candidates.csv"
    )

    all_query_results_df.to_csv(
        combined_csv,
        index=False
    )

    print("\nSaved combined result:", combined_csv)
    display(all_query_results_df)

else:

    print("\nNo query results were generated.")

In [ ]:
# ============================================================
# RANK FRAGMENTS BY SHARED-SIGN SIMILARITY
# CONVNEXT BASE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

MODEL_NAMES = [
    "convnext_base"
]

QUERY_FRAGMENT = "K.1283"
MIN_SHARED_SIGNS = 3


def shared_sign_fragment_similarity(
    fragment_a,
    fragment_b,
    df_valid,
    emb_valid,
    min_shared_signs=3
):
    df_a = df_valid[df_valid["fragmentNumber"] == fragment_a]
    df_b = df_valid[df_valid["fragmentNumber"] == fragment_b]

    shared_signs = set(df_a["signName"]).intersection(
        set(df_b["signName"])
    )

    if len(shared_signs) < min_shared_signs:
        return None

    sign_scores = []

    for sign in shared_signs:

        idx_a = df_a.index[df_a["signName"] == sign].tolist()
        idx_b = df_b.index[df_b["signName"] == sign].tolist()

        emb_a = emb_valid[idx_a]
        emb_b = emb_valid[idx_b]

        sim_matrix = cosine_similarity(
            emb_a,
            emb_b
        )

        sign_scores.append(
            sim_matrix.mean()
        )

    return {
        "fragmentA": fragment_a,
        "fragmentB": fragment_b,
        "num_shared_signs": len(shared_signs),
        "shared_sign_similarity": float(np.mean(sign_scores))
    }


all_candidate_tables = []

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"Ranking shared-sign candidates for {MODEL_NAME}")
    print("=" * 60)

    # ========================================================
    # LOAD SIGN EMBEDDINGS + METADATA
    # ========================================================

    all_embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    emb_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    fragment_info_df = pd.read_csv(
        f"{MODEL_NAME}_fragment_info.csv"
    )

    valid_mask = (
        emb_df["fragmentNumber"].notna() &
        emb_df["signName"].notna()
    )

    df_valid = emb_df.loc[
        valid_mask
    ].reset_index(drop=True)

    emb_valid = all_embeddings[
        valid_mask.values
    ]

    fragment_ids = df_valid["fragmentNumber"].unique()

    if QUERY_FRAGMENT not in fragment_ids:
        print(f"{QUERY_FRAGMENT} not found for {MODEL_NAME}. Skipping.")
        continue

    # ========================================================
    # RANK ALL CANDIDATE FRAGMENTS
    # ========================================================

    results = []

    for candidate in tqdm(
        fragment_ids,
        desc=f"{MODEL_NAME}: Ranking candidates"
    ):

        if candidate == QUERY_FRAGMENT:
            continue

        result = shared_sign_fragment_similarity(
            QUERY_FRAGMENT,
            candidate,
            df_valid,
            emb_valid,
            min_shared_signs=MIN_SHARED_SIGNS
        )

        if result is not None:
            result["model"] = MODEL_NAME
            results.append(result)

    shared_sim_df = pd.DataFrame(results)

    if len(shared_sim_df) == 0:
        print(f"No candidates found for {MODEL_NAME}.")
        continue

    shared_sim_df = shared_sim_df.sort_values(
        "shared_sign_similarity",
        ascending=False
    )

    # ========================================================
    # SAVE RAW SHARED-SIGN RANKING
    # ========================================================

    raw_csv = f"{MODEL_NAME}_{QUERY_FRAGMENT}_shared_sign_candidates.csv"

    shared_sim_df.to_csv(
        raw_csv,
        index=False
    )

    print("Saved:", raw_csv)

    # ========================================================
    # FINAL CANDIDATE TABLE WITH PERIOD AND SIGN COUNTS
    # ========================================================

    candidate_table = shared_sim_df.merge(
        fragment_info_df.rename(columns={
            "fragmentNumber": "fragmentB",
            "period": "candidate_period",
            "num_signs": "candidate_num_signs"
        }),
        on="fragmentB",
        how="left"
    )

    query_info = fragment_info_df[
        fragment_info_df["fragmentNumber"] == QUERY_FRAGMENT
    ]

    if len(query_info) > 0:
        candidate_table["query_period"] = query_info.iloc[0]["period"]
        candidate_table["query_num_signs"] = query_info.iloc[0]["num_signs"]
    else:
        candidate_table["query_period"] = np.nan
        candidate_table["query_num_signs"] = np.nan

    candidate_table["same_period"] = (
        candidate_table["query_period"] ==
        candidate_table["candidate_period"]
    )

    candidate_table = candidate_table[
        [
            "model",
            "fragmentA",
            "fragmentB",
            "shared_sign_similarity",
            "num_shared_signs",
            "query_period",
            "candidate_period",
            "same_period",
            "query_num_signs",
            "candidate_num_signs"
        ]
    ]

    final_csv = f"{MODEL_NAME}_{QUERY_FRAGMENT}_shared_sign_candidate_table.csv"

    candidate_table.to_csv(
        final_csv,
        index=False
    )

    print("Saved:", final_csv)
    display(candidate_table.head(20))

    all_candidate_tables.append(candidate_table)


# ============================================================
# SAVE COMBINED TABLE
# ============================================================

if len(all_candidate_tables) > 0:

    all_candidate_tables_df = pd.concat(
        all_candidate_tables,
        ignore_index=True
    )

    combined_csv = (
        f"convnext_base_{QUERY_FRAGMENT}_shared_sign_candidate_table.csv"
    )

    all_candidate_tables_df.to_csv(
        combined_csv,
        index=False
    )

    print("\nSaved combined candidate table:", combined_csv)
    display(all_candidate_tables_df.head(20))

else:

    print("\nNo shared-sign candidate tables were generated.")

In [ ]:
# ============================================================
# UMAP VISUALIZATION FOR CONVNEXT BASE EMBEDDINGS
# Sign clusters + Tablet/fragment clusters
# ============================================================

# Install once if needed:
# pip install umap-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap

MODEL_NAMES = [
    "convnext_base"
]

MAX_POINTS = 8000
TOP_N_SIGNS = 20
TOP_N_FRAGMENTS = 20
RANDOM_SEED = 42

for MODEL_NAME in MODEL_NAMES:

    print("\n" + "=" * 60)
    print(f"UMAP for {MODEL_NAME}")
    print("=" * 60)

    embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    emb_df = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    valid_mask = (
        emb_df["signName"].notna() &
        emb_df["fragmentNumber"].notna()
    )

    embeddings_valid = embeddings[
        valid_mask.values
    ]

    meta_valid = emb_df.loc[
        valid_mask
    ].reset_index(drop=True)

    # Sample for faster UMAP
    if len(meta_valid) > MAX_POINTS:

        sample_idx = meta_valid.sample(
            n=MAX_POINTS,
            random_state=RANDOM_SEED
        ).index.values

        embeddings_sample = embeddings_valid[
            sample_idx
        ]

        meta_sample = meta_valid.iloc[
            sample_idx
        ].reset_index(drop=True)

    else:

        embeddings_sample = embeddings_valid
        meta_sample = meta_valid

    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=RANDOM_SEED
    )

    umap_coords = reducer.fit_transform(
        embeddings_sample
    )

    meta_sample["umap_x"] = umap_coords[:, 0]
    meta_sample["umap_y"] = umap_coords[:, 1]

    # ========================================================
    # PLOT 1: SIGN CLUSTERS
    # ========================================================

    top_signs = (
        meta_sample["signName"]
        .value_counts()
        .head(TOP_N_SIGNS)
        .index
    )

    sign_plot_df = meta_sample[
        meta_sample["signName"].isin(top_signs)
    ]

    plt.figure(figsize=(9, 7))

    for sign in top_signs:

        sub = sign_plot_df[
            sign_plot_df["signName"] == sign
        ]

        plt.scatter(
            sub["umap_x"],
            sub["umap_y"],
            s=8,
            alpha=0.7,
            label=sign
        )

    plt.title(f"{MODEL_NAME}: UMAP Sign Clusters")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend(
        markerscale=2,
        fontsize=8,
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )
    plt.grid(True)

    plt.savefig(
        f"{MODEL_NAME}_umap_sign_clusters.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    # ========================================================
    # PLOT 2: TABLET / FRAGMENT CLUSTERS
    # ========================================================

    top_fragments = (
        meta_sample["fragmentNumber"]
        .value_counts()
        .head(TOP_N_FRAGMENTS)
        .index
    )

    frag_plot_df = meta_sample[
        meta_sample["fragmentNumber"].isin(top_fragments)
    ]

    plt.figure(figsize=(9, 7))

    for frag in top_fragments:

        sub = frag_plot_df[
            frag_plot_df["fragmentNumber"] == frag
        ]

        plt.scatter(
            sub["umap_x"],
            sub["umap_y"],
            s=8,
            alpha=0.7,
            label=frag
        )

    plt.title(f"{MODEL_NAME}: UMAP Tablet / Fragment Clusters")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend(
        markerscale=2,
        fontsize=8,
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )
    plt.grid(True)

    plt.savefig(
        f"{MODEL_NAME}_umap_tablet_clusters.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    meta_sample.to_csv(
        f"{MODEL_NAME}_umap_coordinates.csv",
        index=False
    )

    print(f"Saved UMAP files for {MODEL_NAME}")

In [ ]:
# ============================================================
# DATASET CLASS
# ============================================================

class SignDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        if "image_path" in self.df.columns:
            img_path = row["image_path"]
        elif "cropPath" in self.df.columns:
            img_path = row["cropPath"]
        else:
            raise ValueError("No image path column found.")

        label = int(row["label_idx"])

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label, img_path

In [ ]:
# ============================================================
# TABLET-HOLDOUT EXPERIMENT — CONVNEXT BASE
# ============================================================

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from tqdm.auto import tqdm
from PIL import Image

# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 42
MIN_SAMPLES_PER_CLASS = 100
BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 1e-4
PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ============================================================
# DATASET CLASS
# ============================================================

class SignDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        if "image_path" in self.df.columns:
            img_path = row["image_path"]
        elif "cropPath" in self.df.columns:
            img_path = row["cropPath"]
        else:
            raise ValueError("No image path column found.")

        label = int(row["label_idx"])

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label, img_path


# ============================================================
# DETECT PATH COLUMN
# ============================================================

if "image_path" in df.columns:
    PATH_COL = "image_path"
elif "cropPath" in df.columns:
    PATH_COL = "cropPath"
else:
    raise ValueError("Expected image_path or cropPath column.")

print("Using path column:", PATH_COL)

# ============================================================
# PREPARE DATAFRAME
# ============================================================

df_holdout_base = df.copy()

df_holdout_base["fragmentNumber"] = df_holdout_base[PATH_COL].apply(
    lambda x: Path(x).stem.split("_")[0]
)

if "label" not in df_holdout_base.columns:

    if "signName" in df_holdout_base.columns:
        SIGN_COL = "signName"
    elif "sign_name" in df_holdout_base.columns:
        SIGN_COL = "sign_name"
    else:
        raise ValueError("No sign column found. Expected signName or sign_name.")

    if "period" not in df_holdout_base.columns:
        raise ValueError("No period column found.")

    df_holdout_base["label"] = (
        df_holdout_base[SIGN_COL].astype(str)
        + "_"
        + df_holdout_base["period"].astype(str)
    )

print("Unique fragments:", df_holdout_base["fragmentNumber"].nunique())
print("Label column ready.")

# ============================================================
# FILTER VALID CLASSES
# ============================================================

class_counts = df_holdout_base["label"].value_counts()

valid_classes = class_counts[
    class_counts >= MIN_SAMPLES_PER_CLASS
].index

df_holdout = df_holdout_base[
    df_holdout_base["label"].isin(valid_classes)
].reset_index(drop=True)

print("\nAfter filtering:")
print("Samples :", len(df_holdout))
print("Classes :", df_holdout["label"].nunique())
print("Tablets :", df_holdout["fragmentNumber"].nunique())

# ============================================================
# TABLET-HOLDOUT SPLIT
# ============================================================

gss1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.35,
    random_state=RANDOM_SEED
)

train_idx, temp_idx = next(
    gss1.split(
        df_holdout,
        groups=df_holdout["fragmentNumber"]
    )
)

train_df = df_holdout.iloc[train_idx].reset_index(drop=True)
temp_df = df_holdout.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=20 / 35,
    random_state=RANDOM_SEED
)

val_idx, test_idx = next(
    gss2.split(
        temp_df,
        groups=temp_df["fragmentNumber"]
    )
)

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

# ============================================================
# KEEP ONLY LABELS PRESENT IN TRAIN
# ============================================================

train_labels = set(train_df["label"].unique())

val_df = val_df[
    val_df["label"].isin(train_labels)
].reset_index(drop=True)

test_df = test_df[
    test_df["label"].isin(train_labels)
].reset_index(drop=True)

# ============================================================
# LABEL ENCODING
# ============================================================

label_to_idx = {
    label: idx
    for idx, label in enumerate(sorted(train_labels))
}

idx_to_label = {
    idx: label
    for label, idx in label_to_idx.items()
}

train_df["label_idx"] = train_df["label"].map(label_to_idx)
val_df["label_idx"] = val_df["label"].map(label_to_idx)
test_df["label_idx"] = test_df["label"].map(label_to_idx)

# ============================================================
# TABLET LEAKAGE CHECK
# ============================================================

train_frags = set(train_df["fragmentNumber"])
val_frags = set(val_df["fragmentNumber"])
test_frags = set(test_df["fragmentNumber"])

print("\n==============================")
print("TABLET LEAKAGE CHECK")
print("==============================")
print("Train-Val overlap :", len(train_frags & val_frags))
print("Train-Test overlap:", len(train_frags & test_frags))
print("Val-Test overlap  :", len(val_frags & test_frags))

assert len(train_frags & val_frags) == 0
assert len(train_frags & test_frags) == 0
assert len(val_frags & test_frags) == 0

print("✓ No tablet leakage detected")

print("\n==============================")
print("TABLET-HOLDOUT SPLIT")
print("==============================")
print("Train samples:", len(train_df))
print("Val samples  :", len(val_df))
print("Test samples :", len(test_df))
print("Train classes:", train_df["label"].nunique())
print("Val classes  :", val_df["label"].nunique())
print("Test classes :", test_df["label"].nunique())
print("Train tablets:", train_df["fragmentNumber"].nunique())
print("Val tablets  :", val_df["fragmentNumber"].nunique())
print("Test tablets :", test_df["fragmentNumber"].nunique())

train_df.to_csv("tablet_holdout_train.csv", index=False)
val_df.to_csv("tablet_holdout_val.csv", index=False)
test_df.to_csv("tablet_holdout_test.csv", index=False)

# ============================================================
# DATASETS + DATALOADERS
# ============================================================

train_dataset = SignDataset(train_df, transform=train_transform)
val_dataset = SignDataset(val_df, transform=test_transform)
test_dataset = SignDataset(test_df, transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)

# ============================================================
# TOP-K ACCURACY
# ============================================================

def top_k_accuracy(outputs, targets, k=1):

    with torch.no_grad():

        _, pred = outputs.topk(k, dim=1)

        correct = pred.eq(
            targets.view(-1, 1).expand_as(pred)
        )

        return correct.sum().item() / targets.size(0)


# ============================================================
# MODEL — CONVNEXT BASE
# ============================================================

MODEL_NAME = "convnext_base_tablet_holdout"

model = models.convnext_base(
    weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1
)

num_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(
    num_features,
    len(label_to_idx)
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.05
)

# ============================================================
# TRAINING LOOP
# ============================================================

history = {
    "train_loss": [],
    "val_loss": [],
    "train_top1": [],
    "train_top2": [],
    "train_top3": [],
    "val_top1": [],
    "val_top2": [],
    "val_top3": [],
}

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

for epoch in range(NUM_EPOCHS):

    model.train()

    train_loss = 0.0
    train_top1 = 0.0
    train_top2 = 0.0
    train_top3 = 0.0
    train_batches = 0

    for images, labels, paths in tqdm(
        train_loader,
        desc=f"{MODEL_NAME} | Epoch {epoch+1}/{NUM_EPOCHS}"
    ):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_top1 += top_k_accuracy(outputs, labels, k=1)
        train_top2 += top_k_accuracy(outputs, labels, k=2)
        train_top3 += top_k_accuracy(outputs, labels, k=3)

        train_batches += 1

    train_loss /= train_batches
    train_top1 /= train_batches
    train_top2 /= train_batches
    train_top3 /= train_batches

    model.eval()

    val_loss = 0.0
    val_top1 = 0.0
    val_top2 = 0.0
    val_top3 = 0.0
    val_batches = 0

    with torch.no_grad():

        for images, labels, paths in tqdm(
            val_loader,
            desc=f"{MODEL_NAME} | Validation {epoch+1}/{NUM_EPOCHS}"
        ):

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            val_top1 += top_k_accuracy(outputs, labels, k=1)
            val_top2 += top_k_accuracy(outputs, labels, k=2)
            val_top3 += top_k_accuracy(outputs, labels, k=3)

            val_batches += 1

    val_loss /= val_batches
    val_top1 /= val_batches
    val_top2 /= val_batches
    val_top3 /= val_batches

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_top1"].append(train_top1)
    history["train_top2"].append(train_top2)
    history["train_top3"].append(train_top3)
    history["val_top1"].append(val_top1)
    history["val_top2"].append(val_top2)
    history["val_top3"].append(val_top3)

    print("\n")
    print(f"Epoch      : {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Train Top-1: {train_top1:.4f}")
    print(f"Train Top-2: {train_top2:.4f}")
    print(f"Train Top-3: {train_top3:.4f}")
    print(f"Val Top-1  : {val_top1:.4f}")
    print(f"Val Top-2  : {val_top2:.4f}")
    print(f"Val Top-3  : {val_top3:.4f}")

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_convnext_base_tablet_holdout_sign_classifier.pth"
        )

        patience_counter = 0
        print("Saved best tablet-holdout ConvNeXt Base model.")

    else:

        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

# ============================================================
# SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(history)
history_df.insert(0, "epoch", range(1, len(history_df) + 1))

history_df.to_csv(
    "convnext_base_tablet_holdout_training_history.csv",
    index=False
)

print("Saved: convnext_base_tablet_holdout_training_history.csv")

# ============================================================
# TEST EVALUATION
# ============================================================

model.load_state_dict(
    torch.load(
        "best_convnext_base_tablet_holdout_sign_classifier.pth",
        map_location=DEVICE
    )
)

model.eval()

all_labels = []
all_preds = []

top1_acc = 0.0
top2_acc = 0.0
top3_acc = 0.0
num_batches = 0

with torch.no_grad():

    for images, labels, paths in tqdm(
        test_loader,
        desc="Testing tablet-holdout ConvNeXt Base"
    ):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

        top1_acc += top_k_accuracy(outputs, labels, k=1)
        top2_acc += top_k_accuracy(outputs, labels, k=2)
        top3_acc += top_k_accuracy(outputs, labels, k=3)

        num_batches += 1

top1_acc /= num_batches
top2_acc /= num_batches
top3_acc /= num_batches

precision = precision_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

tablet_holdout_results = pd.DataFrame([{
    "model": "convnext_base_tablet_holdout",
    "split": "tablet_holdout",
    "best_epoch": best_epoch,
    "top1": top1_acc,
    "top2": top2_acc,
    "top3": top3_acc,
    "precision_macro": precision,
    "recall_macro": recall,
    "macro_f1": macro_f1
}])

tablet_holdout_results.to_csv(
    "convnext_base_tablet_holdout_test_results.csv",
    index=False
)

print("\n================ TEST RESULTS ================")
print(tablet_holdout_results)
print("Saved: convnext_base_tablet_holdout_test_results.csv")

In [ ]:
# ============================================================
# BLOCK 1 — EXTRACT TABLET-HOLDOUT EMBEDDINGS
# CONVNEXT BASE
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from tqdm.auto import tqdm
from pathlib import Path
from torch.utils.data import DataLoader

# ============================================================
# MODEL DICTIONARY
# ============================================================

MODEL_DICT = {
    "convnext_base_tablet_holdout": (
        models.convnext_base,
        "best_convnext_base_tablet_holdout_sign_classifier.pth",
        1024
    ),
}

# ============================================================
# LOAD TABLET-HOLDOUT TEST SPLIT
# ============================================================

test_df = pd.read_csv("tablet_holdout_test.csv")

# Make sure label_idx exists
if "label_idx" not in test_df.columns:
    test_df["label_idx"] = test_df["label"].map(label_to_idx)

test_dataset = SignDataset(
    test_df,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)

# ============================================================
# CONVNEXT EMBEDDING FUNCTION
# ============================================================

def extract_convnext_embedding(model, images):

    x = model.features(images)
    x = model.avgpool(x)
    x = torch.flatten(x, 1)

    return x


# ============================================================
# EXTRACT FOR CONVNEXT BASE
# ============================================================

for MODEL_NAME, (MODEL_CLASS, CKPT_PATH, EMB_DIM) in MODEL_DICT.items():

    print("\n" + "=" * 60)
    print(f"Extracting embeddings for {MODEL_NAME}")
    print("=" * 60)

    model = MODEL_CLASS(weights=None)

    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model.load_state_dict(
        torch.load(
            CKPT_PATH,
            map_location=DEVICE
        )
    )

    model = model.to(DEVICE)
    model.eval()

    all_embeddings = []
    all_true_idx = []
    all_pred_idx = []
    all_confidence = []
    all_paths = []

    with torch.no_grad():

        for images, labels, paths in tqdm(
            test_loader,
            desc=f"Extracting {MODEL_NAME}"
        ):

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            confidence, preds = torch.max(
                probs,
                dim=1
            )

            embeddings = extract_convnext_embedding(
                model,
                images
            )

            all_embeddings.append(
                embeddings.cpu().numpy()
            )

            all_true_idx.extend(
                labels.cpu().numpy()
            )

            all_pred_idx.extend(
                preds.cpu().numpy()
            )

            all_confidence.extend(
                confidence.cpu().numpy()
            )

            all_paths.extend(paths)

    all_embeddings = np.vstack(all_embeddings)
    all_true_idx = np.array(all_true_idx)
    all_pred_idx = np.array(all_pred_idx)
    all_confidence = np.array(all_confidence)

    meta = pd.DataFrame({
        "image_path": all_paths,
        "true_idx": all_true_idx,
        "pred_idx": all_pred_idx,
        "confidence": all_confidence
    })

    meta["true_label"] = meta["true_idx"].map(idx_to_label)
    meta["pred_label"] = meta["pred_idx"].map(idx_to_label)

    meta["correct"] = meta["true_idx"] == meta["pred_idx"]

    meta["fragmentNumber"] = meta["image_path"].apply(
        lambda x: Path(x).stem.split("_")[0]
    )

    meta["signName"] = meta["true_label"].apply(
        lambda x: str(x).split("_")[0]
    )

    meta["period"] = meta["true_label"].apply(
        lambda x: "_".join(str(x).split("_")[1:])
    )

    np.save(
        f"{MODEL_NAME}_test_embeddings.npy",
        all_embeddings
    )

    meta.to_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv",
        index=False
    )

    print("Embedding shape:", all_embeddings.shape)
    print("Expected embedding dim:", EMB_DIM)
    print("Saved:", f"{MODEL_NAME}_test_embeddings.npy")
    print("Saved:", f"{MODEL_NAME}_test_embedding_metadata.csv")

    del model
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# BLOCK 2 — UMAP SIGN + TABLET CLUSTERS
# CONVNEXT BASE TABLET-HOLDOUT
# ============================================================

# Install once if needed:
# pip install umap-learn

import umap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL_LIST = [
    "convnext_base_tablet_holdout"
]

MAX_POINTS = 8000
TOP_N_SIGNS = 20
TOP_N_FRAGMENTS = 20
RANDOM_SEED = 42

for MODEL_NAME in MODEL_LIST:

    print("\n" + "=" * 60)
    print(f"UMAP for {MODEL_NAME}")
    print("=" * 60)

    embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    meta = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    if len(meta) > MAX_POINTS:

        meta_sample = meta.sample(
            n=MAX_POINTS,
            random_state=RANDOM_SEED
        ).reset_index(drop=False)

        embeddings_sample = embeddings[
            meta_sample["index"].values
        ]

    else:

        meta_sample = meta.copy().reset_index(drop=False)
        embeddings_sample = embeddings

    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=RANDOM_SEED
    )

    coords = reducer.fit_transform(
        embeddings_sample
    )

    meta_sample["umap_x"] = coords[:, 0]
    meta_sample["umap_y"] = coords[:, 1]

    meta_sample.to_csv(
        f"{MODEL_NAME}_umap_coordinates.csv",
        index=False
    )

    # ========================================================
    # FIGURE 1: SIGN CLUSTERS
    # ========================================================

    top_signs = (
        meta_sample["signName"]
        .value_counts()
        .head(TOP_N_SIGNS)
        .index
    )

    sign_plot_df = meta_sample[
        meta_sample["signName"].isin(top_signs)
    ]

    plt.figure(figsize=(10, 8))

    for sign in top_signs:

        sub = sign_plot_df[
            sign_plot_df["signName"] == sign
        ]

        plt.scatter(
            sub["umap_x"],
            sub["umap_y"],
            s=8,
            alpha=0.7,
            label=sign
        )

    plt.title(f"{MODEL_NAME}: UMAP Sign Clusters")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend(
        markerscale=2,
        fontsize=8,
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )
    plt.grid(True)

    plt.savefig(
        f"{MODEL_NAME}_figure1_umap_sign_clusters.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    # ========================================================
    # FIGURE 2: TABLET / FRAGMENT CLUSTERS
    # ========================================================

    top_fragments = (
        meta_sample["fragmentNumber"]
        .value_counts()
        .head(TOP_N_FRAGMENTS)
        .index
    )

    frag_plot_df = meta_sample[
        meta_sample["fragmentNumber"].isin(top_fragments)
    ]

    plt.figure(figsize=(10, 8))

    for frag in top_fragments:

        sub = frag_plot_df[
            frag_plot_df["fragmentNumber"] == frag
        ]

        plt.scatter(
            sub["umap_x"],
            sub["umap_y"],
            s=8,
            alpha=0.7,
            label=frag
        )

    plt.title(f"{MODEL_NAME}: UMAP Tablet / Fragment Clusters")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend(
        markerscale=2,
        fontsize=8,
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )
    plt.grid(True)

    plt.savefig(
        f"{MODEL_NAME}_figure2_umap_tablet_clusters.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# ============================================================
# BLOCK 3 — NEAREST NEIGHBORS FROM UNSEEN TABLETS
# CONVNEXT BASE TABLET-HOLDOUT
# ============================================================

from sklearn.neighbors import NearestNeighbors
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODEL_LIST = [
    "convnext_base_tablet_holdout"
]

QUERY_INDEX = 0
TOP_K = 5

for MODEL_NAME in MODEL_LIST:

    print("\n" + "=" * 60)
    print(f"Nearest-neighbor figure for {MODEL_NAME}")
    print("=" * 60)

    embeddings = np.load(
        f"{MODEL_NAME}_test_embeddings.npy"
    )

    meta = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    nn = NearestNeighbors(
        n_neighbors=50,
        metric="cosine",
        algorithm="brute"
    )

    nn.fit(embeddings)

    query_embedding = embeddings[
        QUERY_INDEX
    ].reshape(1, -1)

    query_fragment = meta.loc[
        QUERY_INDEX,
        "fragmentNumber"
    ]

    distances, indices = nn.kneighbors(
        query_embedding
    )

    selected_indices = [QUERY_INDEX]
    selected_sims = [1.0]

    for idx, dist in zip(indices[0], distances[0]):

        if idx == QUERY_INDEX:
            continue

        if meta.loc[idx, "fragmentNumber"] == query_fragment:
            continue

        selected_indices.append(idx)
        selected_sims.append(1.0 - dist)

        if len(selected_indices) == TOP_K + 1:
            break

    plt.figure(figsize=(3 * len(selected_indices), 4))

    for plot_i, idx in enumerate(selected_indices):

        row = meta.loc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        plt.subplot(
            1,
            len(selected_indices),
            plot_i + 1
        )

        plt.imshow(image)
        plt.axis("off")

        if plot_i == 0:

            title = (
                "Query\n"
                f"GT: {row['true_label']}\n"
                f"Tablet: {row['fragmentNumber']}"
            )

        else:

            title = (
                f"NN-{plot_i}\n"
                f"GT: {row['true_label']}\n"
                f"Tablet: {row['fragmentNumber']}\n"
                f"Sim: {selected_sims[plot_i]:.3f}"
            )

        plt.title(
            title,
            fontsize=8
        )

    plt.suptitle(
        f"{MODEL_NAME}: Nearest Neighbors from Different Tablets",
        fontsize=12
    )

    plt.tight_layout()

    plt.savefig(
        f"{MODEL_NAME}_figure3_nearest_neighbors_unseen_tablets.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# ============================================================
# BLOCK 4 — CORRECT VS WRONG PREDICTIONS
# CONVNEXT BASE TABLET-HOLDOUT
# ============================================================

from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

MODEL_LIST = [
    "convnext_base_tablet_holdout"
]

NUM_CASES = 5

for MODEL_NAME in MODEL_LIST:

    print("\n" + "=" * 60)
    print(f"Correct vs wrong examples for {MODEL_NAME}")
    print("=" * 60)

    meta = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    correct_cases = (
        meta[meta["correct"] == True]
        .sort_values("confidence", ascending=False)
        .head(NUM_CASES)
    )

    wrong_cases = (
        meta[meta["correct"] == False]
        .sort_values("confidence", ascending=False)
        .head(NUM_CASES)
    )

    selected = pd.concat([
        correct_cases.assign(case_type="Correct"),
        wrong_cases.assign(case_type="Wrong")
    ]).reset_index(drop=True)

    plt.figure(figsize=(3 * NUM_CASES, 8))

    for i, row in selected.iterrows():

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        plt.subplot(
            2,
            NUM_CASES,
            i + 1
        )

        plt.imshow(image)
        plt.axis("off")

        plt.title(
            f"{row['case_type']}\n"
            f"GT: {row['true_label']}\n"
            f"Pred: {row['pred_label']}\n"
            f"Conf: {row['confidence']:.2f}\n"
            f"Tablet: {row['fragmentNumber']}",
            fontsize=8
        )

    plt.suptitle(
        f"{MODEL_NAME}: Correct vs Wrong Predictions",
        fontsize=12
    )

    plt.tight_layout()

    plt.savefig(
        f"{MODEL_NAME}_figure4_correct_vs_wrong.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# ============================================================
# GRAD-CAM FOR CORRECT AND WRONG CASES
# CONVNEXT BASE TABLET-HOLDOUT
# ============================================================

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ============================================================
# MODEL DICTIONARY
# ============================================================

MODEL_DICT = {
    "convnext_base_tablet_holdout": (
        models.convnext_base,
        "best_convnext_base_tablet_holdout_sign_classifier.pth"
    ),
}

# ============================================================
# IMAGE TRANSFORM
# ============================================================

cam_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

NUM_CORRECT = 5
NUM_WRONG = 5

# ============================================================
# RUN GRAD-CAM
# ============================================================

for MODEL_NAME, (MODEL_CLASS, CKPT_PATH) in MODEL_DICT.items():

    print("\n" + "=" * 60)
    print("Grad-CAM:", MODEL_NAME)
    print("=" * 60)

    OUTPUT_DIR = f"{MODEL_NAME}_gradcam"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    meta = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    # ========================================================
    # LOAD CONVNEXT BASE MODEL
    # ========================================================

    model = MODEL_CLASS(weights=None)

    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model.load_state_dict(
        torch.load(
            CKPT_PATH,
            map_location=DEVICE
        )
    )

    model = model.to(DEVICE)
    model.eval()

    # ========================================================
    # SELECT CORRECT AND WRONG CASES
    # ========================================================

    correct_cases = (
        meta[meta["correct"] == True]
        .sort_values("confidence", ascending=False)
        .head(NUM_CORRECT)
    )

    wrong_cases = (
        meta[meta["correct"] == False]
        .sort_values("confidence", ascending=False)
        .head(NUM_WRONG)
    )

    selected_cases = pd.concat([
        correct_cases.assign(case_type="correct"),
        wrong_cases.assign(case_type="wrong")
    ]).reset_index(drop=True)

    # ========================================================
    # CONVNEXT TARGET LAYER
    # ========================================================

    target_layers = [
        model.features[-1][-1]
    ]

    cam = GradCAM(
        model=model,
        target_layers=target_layers
    )

    # ========================================================
    # GENERATE GRAD-CAM IMAGES
    # ========================================================

    for idx, row in selected_cases.iterrows():

        img_path = row["image_path"]
        pred_idx = int(row["pred_idx"])

        image = Image.open(img_path).convert("RGB")

        input_tensor = cam_transform(image).unsqueeze(0).to(DEVICE)

        rgb_img = image.resize((224, 224))
        rgb_img = np.array(rgb_img).astype(np.float32) / 255.0

        targets = [
            ClassifierOutputTarget(pred_idx)
        ]

        grayscale_cam = cam(
            input_tensor=input_tensor,
            targets=targets
        )[0]

        cam_image = show_cam_on_image(
            rgb_img,
            grayscale_cam,
            use_rgb=True
        )

        plt.figure(figsize=(8, 4))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(cam_image)
        plt.title(
            f"{row['case_type'].upper()}\n"
            f"GT: {row['true_label']}\n"
            f"Pred: {row['pred_label']}\n"
            f"Conf: {row['confidence']:.2f}"
        )
        plt.axis("off")

        plt.tight_layout()

        save_path = os.path.join(
            OUTPUT_DIR,
            f"{MODEL_NAME}_{row['case_type']}_{idx}_gradcam.png"
        )

        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

        print("Saved:", save_path)

    del model
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# OCCLUSION SENSITIVITY MAPS
# CONVNEXT BASE TABLET-HOLDOUT
# ============================================================

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms

# ============================================================
# MODEL DICTIONARY
# ============================================================

MODEL_DICT = {
    "convnext_base_tablet_holdout": (
        models.convnext_base,
        "best_convnext_base_tablet_holdout_sign_classifier.pth"
    ),
}

# ============================================================
# TRANSFORMS
# ============================================================

occ_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

to_tensor_no_norm = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor()
])

# ============================================================
# SETTINGS
# ============================================================

PATCH_SIZE = 32
STRIDE = 16
NUM_CORRECT = 3
NUM_WRONG = 3

mean = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)


def normalize_tensor(x):

    return (x - mean) / std


# ============================================================
# RUN OCCLUSION
# ============================================================

for MODEL_NAME, (MODEL_CLASS, CKPT_PATH) in MODEL_DICT.items():

    print("\n" + "=" * 60)
    print("Occlusion:", MODEL_NAME)
    print("=" * 60)

    OUTPUT_DIR = f"{MODEL_NAME}_occlusion"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    meta = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    # ========================================================
    # LOAD CONVNEXT BASE MODEL
    # ========================================================

    model = MODEL_CLASS(weights=None)

    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model.load_state_dict(
        torch.load(
            CKPT_PATH,
            map_location=DEVICE
        )
    )

    model = model.to(DEVICE)
    model.eval()

    # Move normalization constants to DEVICE
    mean_device = mean.to(DEVICE)
    std_device = std.to(DEVICE)

    def normalize_tensor_device(x):

        return (x - mean_device) / std_device

    # ========================================================
    # SELECT CASES
    # ========================================================

    correct_cases = (
        meta[meta["correct"] == True]
        .sort_values("confidence", ascending=False)
        .head(NUM_CORRECT)
    )

    wrong_cases = (
        meta[meta["correct"] == False]
        .sort_values("confidence", ascending=False)
        .head(NUM_WRONG)
    )

    selected_cases = pd.concat([
        correct_cases.assign(case_type="correct"),
        wrong_cases.assign(case_type="wrong")
    ]).reset_index(drop=True)

    # ========================================================
    # OCCLUSION PER IMAGE
    # ========================================================

    for case_idx, row in selected_cases.iterrows():

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        img_tensor = to_tensor_no_norm(image)  # [0,1], 3x224x224

        input_tensor = normalize_tensor_device(
            img_tensor.clone().to(DEVICE)
        ).unsqueeze(0)

        target_class = int(row["pred_idx"])

        with torch.no_grad():

            output = model(input_tensor)

            prob = torch.softmax(
                output,
                dim=1
            )[0, target_class].item()

        H, W = img_tensor.shape[1], img_tensor.shape[2]

        heatmap = np.zeros(
            (H, W),
            dtype=np.float32
        )

        countmap = np.zeros(
            (H, W),
            dtype=np.float32
        )

        for y in range(0, H - PATCH_SIZE + 1, STRIDE):

            for x in range(0, W - PATCH_SIZE + 1, STRIDE):

                occluded = img_tensor.clone()

                # replace patch with neutral gray
                occluded[
                    :,
                    y:y + PATCH_SIZE,
                    x:x + PATCH_SIZE
                ] = 0.5

                occluded_norm = normalize_tensor_device(
                    occluded.to(DEVICE)
                ).unsqueeze(0)

                with torch.no_grad():

                    out_occ = model(occluded_norm)

                    prob_occ = torch.softmax(
                        out_occ,
                        dim=1
                    )[0, target_class].item()

                drop = prob - prob_occ

                heatmap[
                    y:y + PATCH_SIZE,
                    x:x + PATCH_SIZE
                ] += drop

                countmap[
                    y:y + PATCH_SIZE,
                    x:x + PATCH_SIZE
                ] += 1

        heatmap = heatmap / np.maximum(
            countmap,
            1
        )

        heatmap = np.maximum(
            heatmap,
            0
        )

        if heatmap.max() > 0:

            heatmap = heatmap / heatmap.max()

        rgb_img = np.array(
            image.resize((224, 224))
        ).astype(np.float32) / 255.0

        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plt.imshow(rgb_img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(
            heatmap,
            cmap="hot"
        )
        plt.title("Occlusion sensitivity")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(rgb_img)
        plt.imshow(
            heatmap,
            cmap="hot",
            alpha=0.45
        )
        plt.title(
            f"{row['case_type'].upper()}\n"
            f"GT: {row['true_label']}\n"
            f"Pred: {row['pred_label']}\n"
            f"Conf: {prob:.2f}"
        )
        plt.axis("off")

        plt.tight_layout()

        save_path = os.path.join(
            OUTPUT_DIR,
            f"{MODEL_NAME}_{row['case_type']}_{case_idx}_occlusion.png"
        )

        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

        print("Saved:", save_path)

    del model
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# QUANTITATIVE OCCLUSION BIAS TEST
# CONVNEXT BASE TABLET-HOLDOUT
# ============================================================

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import models, transforms
from tqdm.auto import tqdm

# ============================================================
# MODEL DICTIONARY
# ============================================================

MODEL_DICT = {
    "convnext_base_tablet_holdout": (
        models.convnext_base,
        "best_convnext_base_tablet_holdout_sign_classifier.pth"
    ),
}

# ============================================================
# SETTINGS
# ============================================================

PATCH_SIZE = 32
STRIDE = 16

NUM_CORRECT = 50
NUM_WRONG = 50

IMAGE_SIZE = 224

OUTPUT_CSV = "convnext_base_quantitative_occlusion_bias_results.csv"

# ============================================================
# TRANSFORMS
# ============================================================

to_tensor_no_norm = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor()
])

mean = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)

# ============================================================
# REGION MASKS
# ============================================================

def make_region_masks(image_size=224):

    yy, xx = np.mgrid[0:image_size, 0:image_size]

    # approximate sign area = central 60% of crop
    margin = int(image_size * 0.20)

    center_mask = (
        (xx >= margin) &
        (xx < image_size - margin) &
        (yy >= margin) &
        (yy < image_size - margin)
    )

    border_mask = ~center_mask

    return center_mask, border_mask


center_mask, border_mask = make_region_masks(IMAGE_SIZE)

# ============================================================
# OCCLUSION FUNCTION
# ============================================================

def compute_occlusion_statistics(
    model,
    image_path,
    target_class,
    mean_device,
    std_device,
    patch_size=32,
    stride=16
):

    def normalize_tensor_device(x):
        return (x - mean_device) / std_device

    image = Image.open(
        image_path
    ).convert("RGB")

    img_tensor = to_tensor_no_norm(image)

    input_tensor = normalize_tensor_device(
        img_tensor.clone().to(DEVICE)
    ).unsqueeze(0)

    with torch.no_grad():

        output = model(input_tensor)

        probs = torch.softmax(
            output,
            dim=1
        )

        original_prob = probs[0, target_class].item()

    H, W = img_tensor.shape[1], img_tensor.shape[2]

    heatmap = np.zeros(
        (H, W),
        dtype=np.float32
    )

    countmap = np.zeros(
        (H, W),
        dtype=np.float32
    )

    patch_rows = []

    for y in range(0, H - patch_size + 1, stride):

        for x in range(0, W - patch_size + 1, stride):

            occluded = img_tensor.clone()

            occluded[
                :,
                y:y + patch_size,
                x:x + patch_size
            ] = 0.5

            occluded_norm = normalize_tensor_device(
                occluded.to(DEVICE)
            ).unsqueeze(0)

            with torch.no_grad():

                out_occ = model(occluded_norm)

                prob_occ = torch.softmax(
                    out_occ,
                    dim=1
                )[0, target_class].item()

            drop = original_prob - prob_occ

            heatmap[
                y:y + patch_size,
                x:x + patch_size
            ] += drop

            countmap[
                y:y + patch_size,
                x:x + patch_size
            ] += 1

            patch_center_y = y + patch_size // 2
            patch_center_x = x + patch_size // 2

            is_center_patch = center_mask[
                patch_center_y,
                patch_center_x
            ]

            patch_rows.append({
                "x": x,
                "y": y,
                "drop": float(drop),
                "region": "center" if is_center_patch else "border"
            })

    heatmap = heatmap / np.maximum(
        countmap,
        1
    )

    patch_df = pd.DataFrame(
        patch_rows
    )

    center_drops = patch_df[
        patch_df["region"] == "center"
    ]["drop"]

    border_drops = patch_df[
        patch_df["region"] == "border"
    ]["drop"]

    mean_drop = float(
        patch_df["drop"].mean()
    )

    max_drop = float(
        patch_df["drop"].max()
    )

    positive_mean_drop = float(
        patch_df[patch_df["drop"] > 0]["drop"].mean()
    ) if (patch_df["drop"] > 0).any() else 0.0

    center_mean_drop = float(
        center_drops.mean()
    )

    border_mean_drop = float(
        border_drops.mean()
    )

    center_max_drop = float(
        center_drops.max()
    )

    border_max_drop = float(
        border_drops.max()
    )

    center_positive_mean_drop = float(
        center_drops[center_drops > 0].mean()
    ) if (center_drops > 0).any() else 0.0

    border_positive_mean_drop = float(
        border_drops[border_drops > 0].mean()
    ) if (border_drops > 0).any() else 0.0

    center_minus_border = (
        center_mean_drop - border_mean_drop
    )

    center_border_ratio = (
        center_mean_drop / (border_mean_drop + 1e-8)
    )

    high_impact_threshold = 0.10

    high_impact_patch_ratio = float(
        np.mean(patch_df["drop"] > high_impact_threshold)
    )

    center_high_impact_patch_ratio = float(
        np.mean(center_drops > high_impact_threshold)
    )

    border_high_impact_patch_ratio = float(
        np.mean(border_drops > high_impact_threshold)
    )

    return {
        "original_confidence": original_prob,
        "mean_drop": mean_drop,
        "positive_mean_drop": positive_mean_drop,
        "max_drop": max_drop,
        "center_mean_drop": center_mean_drop,
        "border_mean_drop": border_mean_drop,
        "center_positive_mean_drop": center_positive_mean_drop,
        "border_positive_mean_drop": border_positive_mean_drop,
        "center_max_drop": center_max_drop,
        "border_max_drop": border_max_drop,
        "center_minus_border": center_minus_border,
        "center_border_ratio": center_border_ratio,
        "high_impact_patch_ratio": high_impact_patch_ratio,
        "center_high_impact_patch_ratio": center_high_impact_patch_ratio,
        "border_high_impact_patch_ratio": border_high_impact_patch_ratio
    }


# ============================================================
# RUN OCCLUSION QUANTITATIVE TEST
# ============================================================

all_results = []

for MODEL_NAME, (MODEL_CLASS, CKPT_PATH) in MODEL_DICT.items():

    print("\n" + "=" * 60)
    print(f"Quantitative occlusion test: {MODEL_NAME}")
    print("=" * 60)

    meta = pd.read_csv(
        f"{MODEL_NAME}_test_embedding_metadata.csv"
    )

    # ========================================================
    # LOAD CONVNEXT BASE MODEL
    # ========================================================

    model = MODEL_CLASS(weights=None)

    num_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        num_features,
        len(label_to_idx)
    )

    model.load_state_dict(
        torch.load(
            CKPT_PATH,
            map_location=DEVICE
        )
    )

    model = model.to(DEVICE)
    model.eval()

    mean_device = mean.to(DEVICE)
    std_device = std.to(DEVICE)

    correct_cases = (
        meta[meta["correct"] == True]
        .sort_values("confidence", ascending=False)
        .head(NUM_CORRECT)
    )

    wrong_cases = (
        meta[meta["correct"] == False]
        .sort_values("confidence", ascending=False)
        .head(NUM_WRONG)
    )

    selected_cases = pd.concat([
        correct_cases.assign(case_type="correct"),
        wrong_cases.assign(case_type="wrong")
    ]).reset_index(drop=True)

    for case_idx, row in tqdm(
        selected_cases.iterrows(),
        total=len(selected_cases),
        desc=MODEL_NAME
    ):

        target_class = int(row["pred_idx"])

        stats = compute_occlusion_statistics(
            model=model,
            image_path=row["image_path"],
            target_class=target_class,
            mean_device=mean_device,
            std_device=std_device,
            patch_size=PATCH_SIZE,
            stride=STRIDE
        )

        result = {
            "model": MODEL_NAME,
            "case_idx": case_idx,
            "case_type": row["case_type"],
            "image_path": row["image_path"],
            "true_label": row["true_label"],
            "pred_label": row["pred_label"],
            "correct": bool(row["correct"]),
            "confidence_from_metadata": float(row["confidence"]),
        }

        result.update(stats)

        all_results.append(result)

    del model
    torch.cuda.empty_cache()

# ============================================================
# SAVE FULL RESULTS
# ============================================================

occlusion_df = pd.DataFrame(all_results)

occlusion_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:", OUTPUT_CSV)

# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = (
    occlusion_df
    .groupby(["model", "case_type"])
    [
        [
            "original_confidence",
            "mean_drop",
            "positive_mean_drop",
            "max_drop",
            "center_mean_drop",
            "border_mean_drop",
            "center_positive_mean_drop",
            "border_positive_mean_drop",
            "center_minus_border",
            "center_border_ratio",
            "high_impact_patch_ratio",
            "center_high_impact_patch_ratio",
            "border_high_impact_patch_ratio"
        ]
    ]
    .mean()
    .reset_index()
)

summary_df.to_csv(
    "convnext_base_quantitative_occlusion_bias_summary.csv",
    index=False
)

print("\nOcclusion summary:")
print(summary_df)
print("\nSaved: convnext_base_quantitative_occlusion_bias_summary.csv")

In [ ]:
# ============================================================
# SAME-SIGN SIMILARITY BY TABLET AND PERIOD
# CONVNEXT BASE
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS
# ============================================================

MODEL_NAME = "convnext_base"

EMBEDDINGS_PATH = f"{MODEL_NAME}_test_embeddings.npy"
METADATA_PATH = f"{MODEL_NAME}_test_embedding_metadata.csv"

MAX_PAIRS_PER_SIGN = 500
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ============================================================
# LOAD EMBEDDINGS + METADATA
# ============================================================

embeddings = np.load(
    EMBEDDINGS_PATH
)

emb_df = pd.read_csv(
    METADATA_PATH
)

# ============================================================
# VALID ROWS ONLY
# ============================================================

valid_mask = (
    emb_df["signName"].notna() &
    emb_df["fragmentNumber"].notna() &
    emb_df["period"].notna()
)

df_valid = emb_df.loc[
    valid_mask
].reset_index(drop=True)

emb_valid = embeddings[
    valid_mask.values
]

print("Valid samples:", len(df_valid))
print("Embedding shape:", emb_valid.shape)
print("Unique signs:", df_valid["signName"].nunique())
print("Unique tablets:", df_valid["fragmentNumber"].nunique())
print("Unique periods:", df_valid["period"].nunique())

# ============================================================
# SIMILARITY GROUPS
# ============================================================

same_tablet_sims = []
diff_tablet_same_period_sims = []
diff_tablet_diff_period_sims = []

pair_rows = []

# ============================================================
# MAIN ANALYSIS
# ============================================================

for sign in tqdm(
    df_valid["signName"].unique(),
    desc=f"{MODEL_NAME}: same-sign similarity by period"
):

    sign_idx = df_valid.index[
        df_valid["signName"] == sign
    ].tolist()

    if len(sign_idx) < 2:
        continue

    total_possible_pairs = len(sign_idx) * (len(sign_idx) - 1) // 2

    n_pairs = min(
        MAX_PAIRS_PER_SIGN,
        total_possible_pairs
    )

    sampled_pairs = set()

    attempts = 0
    max_attempts = n_pairs * 20

    while len(sampled_pairs) < n_pairs and attempts < max_attempts:

        i, j = random.sample(sign_idx, 2)

        if i > j:
            i, j = j, i

        pair = (i, j)

        if pair in sampled_pairs:
            attempts += 1
            continue

        sampled_pairs.add(pair)
        attempts += 1

        sim = cosine_similarity(
            emb_valid[i].reshape(1, -1),
            emb_valid[j].reshape(1, -1)
        )[0, 0]

        frag_i = df_valid.loc[i, "fragmentNumber"]
        frag_j = df_valid.loc[j, "fragmentNumber"]

        period_i = df_valid.loc[i, "period"]
        period_j = df_valid.loc[j, "period"]

        if frag_i == frag_j:

            group = "same_tablet"
            same_tablet_sims.append(sim)

        elif period_i == period_j:

            group = "different_tablet_same_period"
            diff_tablet_same_period_sims.append(sim)

        else:

            group = "different_tablet_different_period"
            diff_tablet_diff_period_sims.append(sim)

        pair_rows.append({
            "model": MODEL_NAME,
            "signName": sign,
            "idx_i": i,
            "idx_j": j,
            "fragment_i": frag_i,
            "fragment_j": frag_j,
            "period_i": period_i,
            "period_j": period_j,
            "group": group,
            "cosine_similarity": float(sim)
        })

# ============================================================
# SAFE MEAN FUNCTION
# ============================================================

def safe_mean(values):

    if len(values) == 0:
        return np.nan

    return float(np.mean(values))


# ============================================================
# SUMMARY RESULTS
# ============================================================

same_tablet_mean = safe_mean(
    same_tablet_sims
)

diff_tablet_same_period_mean = safe_mean(
    diff_tablet_same_period_sims
)

diff_tablet_diff_period_mean = safe_mean(
    diff_tablet_diff_period_sims
)

summary_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "same_tablet_mean": same_tablet_mean,
    "different_tablet_same_period_mean": diff_tablet_same_period_mean,
    "different_tablet_different_period_mean": diff_tablet_diff_period_mean,
    "same_tablet_minus_same_period": same_tablet_mean - diff_tablet_same_period_mean,
    "same_period_minus_different_period": diff_tablet_same_period_mean - diff_tablet_diff_period_mean,
    "same_tablet_pairs": len(same_tablet_sims),
    "different_tablet_same_period_pairs": len(diff_tablet_same_period_sims),
    "different_tablet_different_period_pairs": len(diff_tablet_diff_period_sims),
    "total_pairs": len(pair_rows)
}])

print("\n==============================")
print("SAME-SIGN SIMILARITY BY PERIOD")
print("==============================")
print(summary_df)

# ============================================================
# SAVE RESULTS
# ============================================================

pair_df = pd.DataFrame(
    pair_rows
)

pair_csv = f"{MODEL_NAME}_same_sign_similarity_by_period_pairs.csv"
summary_csv = f"{MODEL_NAME}_same_sign_similarity_by_period_summary.csv"

pair_df.to_csv(
    pair_csv,
    index=False
)

summary_df.to_csv(
    summary_csv,
    index=False
)

print("\nSaved:", pair_csv)
print("Saved:", summary_csv)

# ============================================================
# GROUP-WISE DISTRIBUTION SUMMARY
# ============================================================

distribution_df = (
    pair_df
    .groupby("group")["cosine_similarity"]
    .agg(["count", "mean", "std", "median", "min", "max"])
    .reset_index()
)

distribution_csv = f"{MODEL_NAME}_same_sign_similarity_by_period_distribution.csv"

distribution_df.to_csv(
    distribution_csv,
    index=False
)

print("Saved:", distribution_csv)
print("\nDistribution summary:")
print(distribution_df)

# ============================================================
# PLOT 1 — HISTOGRAM DISTRIBUTIONS
# ============================================================

plt.figure(figsize=(9, 6))

if len(same_tablet_sims) > 0:
    plt.hist(
        same_tablet_sims,
        bins=50,
        alpha=0.5,
        density=True,
        label="Same tablet"
    )

if len(diff_tablet_same_period_sims) > 0:
    plt.hist(
        diff_tablet_same_period_sims,
        bins=50,
        alpha=0.5,
        density=True,
        label="Different tablet, same period"
    )

if len(diff_tablet_diff_period_sims) > 0:
    plt.hist(
        diff_tablet_diff_period_sims,
        bins=50,
        alpha=0.5,
        density=True,
        label="Different tablet, different period"
    )

plt.xlabel("Cosine Similarity")
plt.ylabel("Density")
plt.title(f"{MODEL_NAME}: Same-Sign Similarity by Tablet/Period")
plt.legend()
plt.grid(True)

hist_path = f"{MODEL_NAME}_same_sign_similarity_by_period_histogram.png"

plt.savefig(
    hist_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", hist_path)

# ============================================================
# PLOT 2 — MEAN SIMILARITY BAR PLOT
# ============================================================

bar_df = pd.DataFrame({
    "group": [
        "Same tablet",
        "Diff tablet\nsame period",
        "Diff tablet\ndiff period"
    ],
    "mean_similarity": [
        same_tablet_mean,
        diff_tablet_same_period_mean,
        diff_tablet_diff_period_mean
    ]
})

plt.figure(figsize=(8, 5))

bars = plt.bar(
    bar_df["group"],
    bar_df["mean_similarity"]
)

plt.ylabel("Mean Cosine Similarity")
plt.title(f"{MODEL_NAME}: Mean Same-Sign Similarity")

for bar, value in zip(bars, bar_df["mean_similarity"]):

    if not np.isnan(value):

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.01,
            f"{value:.3f}",
            ha="center"
        )

plt.ylim(0, 1.0)
plt.grid(axis="y")

bar_path = f"{MODEL_NAME}_same_sign_similarity_by_period_barplot.png"

plt.savefig(
    bar_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", bar_path)

In [ ]:
# ============================================================
# SAME-SIGN SIMILARITY BY TABLET AND PERIOD with tablet held out
# CONVNEXT BASE
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS
# ============================================================

MODEL_NAME = "convnext_base_tablet_holdout"

EMBEDDINGS_PATH = f"{MODEL_NAME}_test_embeddings.npy"
METADATA_PATH = f"{MODEL_NAME}_test_embedding_metadata.csv"

MAX_PAIRS_PER_SIGN = 500
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ============================================================
# LOAD EMBEDDINGS + METADATA
# ============================================================

embeddings = np.load(
    EMBEDDINGS_PATH
)

emb_df = pd.read_csv(
    METADATA_PATH
)

# ============================================================
# VALID ROWS ONLY
# ============================================================

valid_mask = (
    emb_df["signName"].notna() &
    emb_df["fragmentNumber"].notna() &
    emb_df["period"].notna()
)

df_valid = emb_df.loc[
    valid_mask
].reset_index(drop=True)

emb_valid = embeddings[
    valid_mask.values
]

print("Valid samples:", len(df_valid))
print("Embedding shape:", emb_valid.shape)
print("Unique signs:", df_valid["signName"].nunique())
print("Unique tablets:", df_valid["fragmentNumber"].nunique())
print("Unique periods:", df_valid["period"].nunique())

# ============================================================
# SIMILARITY GROUPS
# ============================================================

same_tablet_sims = []
diff_tablet_same_period_sims = []
diff_tablet_diff_period_sims = []

pair_rows = []

# ============================================================
# MAIN ANALYSIS
# ============================================================

for sign in tqdm(
    df_valid["signName"].unique(),
    desc=f"{MODEL_NAME}: same-sign similarity by period"
):

    sign_idx = df_valid.index[
        df_valid["signName"] == sign
    ].tolist()

    if len(sign_idx) < 2:
        continue

    total_possible_pairs = len(sign_idx) * (len(sign_idx) - 1) // 2

    n_pairs = min(
        MAX_PAIRS_PER_SIGN,
        total_possible_pairs
    )

    sampled_pairs = set()

    attempts = 0
    max_attempts = n_pairs * 20

    while len(sampled_pairs) < n_pairs and attempts < max_attempts:

        i, j = random.sample(sign_idx, 2)

        if i > j:
            i, j = j, i

        pair = (i, j)

        if pair in sampled_pairs:
            attempts += 1
            continue

        sampled_pairs.add(pair)
        attempts += 1

        sim = cosine_similarity(
            emb_valid[i].reshape(1, -1),
            emb_valid[j].reshape(1, -1)
        )[0, 0]

        frag_i = df_valid.loc[i, "fragmentNumber"]
        frag_j = df_valid.loc[j, "fragmentNumber"]

        period_i = df_valid.loc[i, "period"]
        period_j = df_valid.loc[j, "period"]

        if frag_i == frag_j:

            group = "same_tablet"
            same_tablet_sims.append(sim)

        elif period_i == period_j:

            group = "different_tablet_same_period"
            diff_tablet_same_period_sims.append(sim)

        else:

            group = "different_tablet_different_period"
            diff_tablet_diff_period_sims.append(sim)

        pair_rows.append({
            "model": MODEL_NAME,
            "signName": sign,
            "idx_i": i,
            "idx_j": j,
            "fragment_i": frag_i,
            "fragment_j": frag_j,
            "period_i": period_i,
            "period_j": period_j,
            "group": group,
            "cosine_similarity": float(sim)
        })

# ============================================================
# SAFE MEAN FUNCTION
# ============================================================

def safe_mean(values):

    if len(values) == 0:
        return np.nan

    return float(np.mean(values))


# ============================================================
# SUMMARY RESULTS
# ============================================================

same_tablet_mean = safe_mean(
    same_tablet_sims
)

diff_tablet_same_period_mean = safe_mean(
    diff_tablet_same_period_sims
)

diff_tablet_diff_period_mean = safe_mean(
    diff_tablet_diff_period_sims
)

summary_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "same_tablet_mean": same_tablet_mean,
    "different_tablet_same_period_mean": diff_tablet_same_period_mean,
    "different_tablet_different_period_mean": diff_tablet_diff_period_mean,
    "same_tablet_minus_same_period": same_tablet_mean - diff_tablet_same_period_mean,
    "same_period_minus_different_period": diff_tablet_same_period_mean - diff_tablet_diff_period_mean,
    "same_tablet_pairs": len(same_tablet_sims),
    "different_tablet_same_period_pairs": len(diff_tablet_same_period_sims),
    "different_tablet_different_period_pairs": len(diff_tablet_diff_period_sims),
    "total_pairs": len(pair_rows)
}])

print("\n==============================")
print("SAME-SIGN SIMILARITY BY PERIOD")
print("==============================")
print(summary_df)

# ============================================================
# SAVE RESULTS
# ============================================================

pair_df = pd.DataFrame(
    pair_rows
)

pair_csv = f"{MODEL_NAME}_same_sign_similarity_by_period_pairs.csv"
summary_csv = f"{MODEL_NAME}_same_sign_similarity_by_period_summary.csv"

pair_df.to_csv(
    pair_csv,
    index=False
)

summary_df.to_csv(
    summary_csv,
    index=False
)

print("\nSaved:", pair_csv)
print("Saved:", summary_csv)

# ============================================================
# GROUP-WISE DISTRIBUTION SUMMARY
# ============================================================

distribution_df = (
    pair_df
    .groupby("group")["cosine_similarity"]
    .agg(["count", "mean", "std", "median", "min", "max"])
    .reset_index()
)

distribution_csv = f"{MODEL_NAME}_same_sign_similarity_by_period_distribution.csv"

distribution_df.to_csv(
    distribution_csv,
    index=False
)

print("Saved:", distribution_csv)
print("\nDistribution summary:")
print(distribution_df)

# ============================================================
# PLOT 1 — HISTOGRAM DISTRIBUTIONS
# ============================================================

plt.figure(figsize=(9, 6))

if len(same_tablet_sims) > 0:
    plt.hist(
        same_tablet_sims,
        bins=50,
        alpha=0.5,
        density=True,
        label="Same tablet"
    )

if len(diff_tablet_same_period_sims) > 0:
    plt.hist(
        diff_tablet_same_period_sims,
        bins=50,
        alpha=0.5,
        density=True,
        label="Different tablet, same period"
    )

if len(diff_tablet_diff_period_sims) > 0:
    plt.hist(
        diff_tablet_diff_period_sims,
        bins=50,
        alpha=0.5,
        density=True,
        label="Different tablet, different period"
    )

plt.xlabel("Cosine Similarity")
plt.ylabel("Density")
plt.title(f"{MODEL_NAME}: Same-Sign Similarity by Tablet/Period")
plt.legend()
plt.grid(True)

hist_path = f"{MODEL_NAME}_same_sign_similarity_by_period_histogram.png"

plt.savefig(
    hist_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", hist_path)

# ============================================================
# PLOT 2 — MEAN SIMILARITY BAR PLOT
# ============================================================

bar_df = pd.DataFrame({
    "group": [
        "Same tablet",
        "Diff tablet\nsame period",
        "Diff tablet\ndiff period"
    ],
    "mean_similarity": [
        same_tablet_mean,
        diff_tablet_same_period_mean,
        diff_tablet_diff_period_mean
    ]
})

plt.figure(figsize=(8, 5))

bars = plt.bar(
    bar_df["group"],
    bar_df["mean_similarity"]
)

plt.ylabel("Mean Cosine Similarity")
plt.title(f"{MODEL_NAME}: Mean Same-Sign Similarity")

for bar, value in zip(bars, bar_df["mean_similarity"]):

    if not np.isnan(value):

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.01,
            f"{value:.3f}",
            ha="center"
        )

plt.ylim(0, 1.0)
plt.grid(axis="y")

bar_path = f"{MODEL_NAME}_same_sign_similarity_by_period_barplot.png"

plt.savefig(
    bar_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", bar_path)

In [ ]:
# ============================================================
# CELL 18 — JOIN RETRIEVAL METRICS
# ============================================================

valid_join_eval = join_eval_df[join_eval_df["found"] == True].copy()

recall_at_1 = np.mean(valid_join_eval["rank"] <= 1)
recall_at_5 = np.mean(valid_join_eval["rank"] <= 5)
recall_at_10 = np.mean(valid_join_eval["rank"] <= 10)

mrr = np.mean(1 / valid_join_eval["rank"])

print(f"Recall@1  : {recall_at_1:.4f}")
print(f"Recall@5  : {recall_at_5:.4f}")
print(f"Recall@10 : {recall_at_10:.4f}")
print(f"MRR       : {mrr:.4f}")

In [ ]:
# ============================================================
# CELL 19 — SAVE JOIN EVALUATION
# ============================================================

join_eval_df.to_csv(
    f"{MODEL_NAME}_known_join_evaluation.csv",
    index=False
)

join_metrics_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "num_known_joins": len(known_joins),
    "recall_at_1": recall_at_1,
    "recall_at_5": recall_at_5,
    "recall_at_10": recall_at_10,
    "mrr": mrr
}])

join_metrics_df.to_csv(
    f"{MODEL_NAME}_known_join_metrics.csv",
    index=False
)

print("Saved known join evaluation and metrics.")